## 1. Librerías

In [1]:
# Librerías

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter

from scipy.stats import poisson
import statsmodels.api as sm

## 2. Cargar datos

In [2]:
# Cargar datos

df = pd.read_csv("results.csv")

print("Shape inicial:", df.shape)
print(df.columns.tolist())

df.head()

Shape inicial: (49547, 9)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


## 3. Limpieza

In [3]:
# Limpieza

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["home_score"] = pd.to_numeric(df["home_score"], errors="coerce")
df["away_score"] = pd.to_numeric(df["away_score"], errors="coerce")

df = df.dropna(
    subset=[
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
).copy()

df = df.sort_values("date").reset_index(drop=True)


df["neutral"] = (
    df["neutral"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({"TRUE": 1, "FALSE": 0})
)

if df["neutral"].isna().any():
    raise ValueError("Hay valores no reconocidos en 'neutral'.")

print("Shape después de limpieza:", df.shape)
print(df["neutral"].value_counts())

Shape después de limpieza: (49547, 9)
neutral
0    36389
1    13158
Name: count, dtype: int64


## 4. Resultado H/D/A

In [4]:
# Resultado H/D/A

def get_outcome(row):
    if row["home_score"] > row["away_score"]:
        return "H"
    elif row["home_score"] == row["away_score"]:
        return "D"
    else:
        return "A"

df["outcome"] = df.apply(get_outcome, axis=1)

print(df["outcome"].value_counts())

outcome
H    24276
A    14010
D    11261
Name: count, dtype: int64


# 5. Elo

Parámetros del artículo:
- Elo inicial = 1500.
- Ventaja local = 100 en sedes no neutrales.
- Ventaja = 0 en sedes neutrales.
- K = 60 para FIFA World Cup final.
- K = 35 para campeonatos continentales.
- K = 25 para clasificatorias.
- K = 20 para amistosos y demás.

In [5]:
# Elo

INITIAL_ELO = 1500
HOME_ADVANTAGE = 100

In [6]:
# Elo

def get_k_factor(tournament):
    tournament = str(tournament).strip()
    tournament_lower = tournament.lower()

    if tournament == "FIFA World Cup":
        return 60

    continental = [
        "UEFA Euro",
        "Copa América",
        "African Cup of Nations",
        "AFC Asian Cup",
        "CONCACAF Championship",
        "Gold Cup",
        "Oceania Nations Cup",
    ]

    if tournament in continental:
        return 35

    if (
        "qualification" in tournament_lower
        or "qualifying" in tournament_lower
    ):
        return 25

    return 20

In [7]:
# Elo

def goal_difference_multiplier(goal_diff):
    if goal_diff <= 1:
        return 1.0
    elif goal_diff == 2:
        return 1.5
    else:
        return (11 + goal_diff) / 8

In [8]:
# Elo

def update_elo(
    home_elo,
    away_elo,
    home_goals,
    away_goals,
    k_factor,
    neutral
):
    home_advantage = 0 if neutral else HOME_ADVANTAGE

    expected_home = 1 / (
        1 + 10 ** (
            -(home_elo + home_advantage - away_elo) / 400
        )
    )

    if home_goals > away_goals:
        actual_home = 1.0
    elif home_goals == away_goals:
        actual_home = 0.5
    else:
        actual_home = 0.0

    goal_diff = abs(home_goals - away_goals)
    gamma = goal_difference_multiplier(goal_diff)

    change = k_factor * gamma * (actual_home - expected_home)

    return home_elo + change, away_elo - change

## 6. Historial Elo partido por partido

In [9]:
# Historial Elo partido por partido

elo_ratings = {}

pre_home_elo = []
pre_away_elo = []

post_home_elo = []
post_away_elo = []

for _, row in df.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    if home not in elo_ratings:
        elo_ratings[home] = INITIAL_ELO

    if away not in elo_ratings:
        elo_ratings[away] = INITIAL_ELO

    home_before = elo_ratings[home]
    away_before = elo_ratings[away]

    pre_home_elo.append(home_before)
    pre_away_elo.append(away_before)

    k = get_k_factor(row["tournament"])

    home_after, away_after = update_elo(
        home_before,
        away_before,
        row["home_score"],
        row["away_score"],
        k,
        row["neutral"]
    )

    elo_ratings[home] = home_after
    elo_ratings[away] = away_after

    post_home_elo.append(home_after)
    post_away_elo.append(away_after)

In [10]:
# Historial Elo partido por partido

df["home_elo"] = pre_home_elo
df["away_elo"] = pre_away_elo

df["home_elo_post"] = post_home_elo
df["away_elo_post"] = post_away_elo

df["elo_diff"] = df["home_elo"] - df["away_elo"]

df[
    [
        "date",
        "home_team",
        "away_team",
        "home_elo",
        "away_elo",
        "elo_diff"
    ]
].head(10)

,date,home_team,away_team,home_elo,away_elo,elo_diff
0,1872-11-30,Scotland,England,1500.000000,1500.000000,0.000000
1,1873-03-08,England,Scotland,1502.801300,1497.198700,5.602600
2,1874-03-07,Scotland,England,1486.622531,1513.377469,-26.754938
3,1875-03-06,England,Scotland,1505.454946,1494.545054,10.909891
4,1876-03-04,Scotland,England,1497.633108,1502.366892,-4.733783
5,1876-03-25,Scotland,Wales,1510.451387,1500.000000,10.451387
6,1877-03-03,England,Scotland,1489.548613,1523.433679,-33.885066
7,1877-03-05,Wales,Scotland,1487.017707,1541.254128,-54.236420
8,1878-03-02,Scotland,England,1558.218553,1471.728165,86.490388
9,1878-03-23,Scotland,Wales,1568.407885,1470.053282,98.354603


# 7. Historial mensual de Elo

**Corrección importante:** `m0` se obtiene directamente del Elo pre-partido de la fila actual. No usamos el Elo post-partido del mismo mes para `m0`.

Para `m1`–`m5`, si un equipo no jugó exactamente en ese mes, conservamos su último Elo conocido hasta ese mes.

In [11]:
# Historial mensual de Elo

df["month"] = df["date"].dt.to_period("M")

monthly_elo = {}

for _, row in df.iterrows():

    month = row["month"]

    if month not in monthly_elo:
        monthly_elo[month] = {}

    monthly_elo[month][row["home_team"]] = row["home_elo_post"]
    monthly_elo[month][row["away_team"]] = row["away_elo_post"]

In [12]:
# Historial mensual de Elo

def get_team_elo_for_month(team, month):

    available_months = [
        m for m in monthly_elo.keys()
        if m <= month and team in monthly_elo[m]
    ]

    if not available_months:
        return np.nan

    last_month = max(available_months)

    return monthly_elo[last_month][team]

## 8. Construcción de los seis lags

In [13]:
# M0 = Elo pre-partido

def build_lagged_elo_features(row, n_lags=6):

    current_month = row["month"]

    values = []


    values.append(
        row["home_elo"] - row["away_elo"]
    )


    for lag in range(1, n_lags):

        target_month = current_month - lag

        home_elo = get_team_elo_for_month(
            row["home_team"],
            target_month
        )

        away_elo = get_team_elo_for_month(
            row["away_team"],
            target_month
        )

        if pd.isna(home_elo) or pd.isna(away_elo):
            values.append(np.nan)
        else:
            values.append(home_elo - away_elo)

    return values

In [14]:
# HISTORIAL MENSUAL DE ELO - VERSION OPTIMIZADA

from bisect import bisect_right




team_monthly_elo = {}

for month, teams in monthly_elo.items():

    for team, elo in teams.items():

        if team not in team_monthly_elo:
            team_monthly_elo[team] = {}

        team_monthly_elo[team][month] = elo





team_monthly_elo_fast = {}

for team, history in team_monthly_elo.items():

    months = sorted(history.keys())
    values = [history[m] for m in months]

    team_monthly_elo_fast[team] = (
        months,
        values
    )


def get_team_elo_for_month(team, month):

    if team not in team_monthly_elo_fast:
        return np.nan

    months, values = team_monthly_elo_fast[team]


    idx = bisect_right(months, month) - 1

    if idx < 0:
        return np.nan

    return values[idx]

In [15]:
# CONSTRUCCION DE LOS 6 LAGS DE ELO

elo_lag_cols = [
    "elo_diff_m0",
    "elo_diff_m1",
    "elo_diff_m2",
    "elo_diff_m3",
    "elo_diff_m4",
    "elo_diff_m5"
]


def build_lagged_elo_features(row, n_lags=6):

    current_month = row["month"]

    values = []









    values.append(
        row["home_elo"] - row["away_elo"]
    )








    for lag in range(1, n_lags):

        target_month = current_month - lag

        home_elo = get_team_elo_for_month(
            row["home_team"],
            target_month
        )

        away_elo = get_team_elo_for_month(
            row["away_team"],
            target_month
        )

        if pd.isna(home_elo) or pd.isna(away_elo):

            values.append(np.nan)

        else:

            values.append(
                home_elo - away_elo
            )

    return values



lagged_elo = df.apply(
    build_lagged_elo_features,
    axis=1,
    result_type="expand"
)

lagged_elo.columns = elo_lag_cols




df = df.drop(
    columns=elo_lag_cols,
    errors="ignore"
)



df = pd.concat(
    [df, lagged_elo],
    axis=1
)

print("Lags construidos correctamente.")

Lags construidos correctamente.


In [16]:
# COMPROBACION DE LOS LAGS

print("Valores disponibles por lag:\n")

print(
    df[elo_lag_cols]
    .notna()
    .sum()
)

print("\n" + "=" * 50)

print("Filas completas:")

complete_rows = (
    df[elo_lag_cols]
    .notna()
    .all(axis=1)
    .sum()
)

print(complete_rows)

Valores disponibles por lag:

elo_diff_m0    49547
elo_diff_m1    49022
elo_diff_m2    48937
elo_diff_m3    48916
elo_diff_m4    48898
elo_diff_m5    48878
dtype: int64

Filas completas:
48878


In [17]:
# DIAGNOSTICO POR AÑO

df["year"] = df["date"].dt.year

print("Filas completas por año:\n")

print(
    df.groupby("year")[elo_lag_cols]
      .apply(
          lambda x:
          x.notna()
           .all(axis=1)
           .sum()
      )
      .tail(30)
)

Filas completas por año:

year
1997     899
1998     758
1999     758
2000    1039
2001    1021
2002     767
2003     942
2004    1078
2005     795
2006     830
2007     983
2008    1090
2009     922
2010     853
2011    1117
2012    1009
2013     957
2014     843
2015    1027
2016     904
2017     924
2018     908
2019    1138
2020     347
2021    1115
2022     961
2023    1049
2024    1231
2025    1000
2026     447
dtype: int64


## 9. Diagnóstico de los lags

Si el número de filas completas es muy pequeño, **no continuar ciegamente** con SDR. Esta salida permite detectar si el problema está concentrado en los primeros años o si todavía existe un problema en la construcción mensual.

In [18]:
# Diagnóstico de los lags

df["year"] = df["date"].dt.year

print("Filas completas por año:")
print(
    df.groupby("year")[elo_lag_cols]
      .apply(lambda x: x.notna().all(axis=1).sum())
      .tail(30)
)

print("Valores no nulos por año:")
print(
    df.groupby("year")[elo_lag_cols]
      .apply(lambda x: x.notna().sum())
      .tail(30)
)

Filas completas por año:
year
1997     899
1998     758
1999     758
2000    1039
2001    1021
2002     767
2003     942
2004    1078
2005     795
2006     830
2007     983
2008    1090
2009     922
2010     853
2011    1117
2012    1009
2013     957
2014     843
2015    1027
2016     904
2017     924
2018     908
2019    1138
2020     347
2021    1115
2022     961
2023    1049
2024    1231
2025    1000
2026     447
dtype: int64
Valores no nulos por año:
      elo_diff_m0  elo_diff_m1  elo_diff_m2  elo_diff_m3  elo_diff_m4  \
year                                                                    
1997          907          904          899          899          899   
1998          761          759          758          758          758   
1999          776          764          758          758          758   
2000         1040         1039         1039         1039         1039   
2001         1032         1022         1021         1021         1021   
2002          768          767

# 10. Variables rolling

Se calculan usando solamente los seis partidos anteriores de cada equipo:
- goles anotados;
- goles recibidos;
- puntos promedio;
- diferencia de forma.

El partido actual se añade al historial después de calcular las variables.

In [19]:
# Variables rolling

team_history = {}

home_goals_for = []
home_goals_against = []

away_goals_for = []
away_goals_against = []

home_form = []
away_form = []

for _, row in df.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    if home not in team_history:
        team_history[home] = []

    if away not in team_history:
        team_history[away] = []

    h_history = team_history[home][-6:]
    a_history = team_history[away][-6:]

    if len(h_history) > 0:
        h_gf = np.mean([x["gf"] for x in h_history])
        h_ga = np.mean([x["ga"] for x in h_history])
        h_form = np.mean([x["points"] for x in h_history])
    else:
        h_gf = np.nan
        h_ga = np.nan
        h_form = np.nan

    if len(a_history) > 0:
        a_gf = np.mean([x["gf"] for x in a_history])
        a_ga = np.mean([x["ga"] for x in a_history])
        a_form = np.mean([x["points"] for x in a_history])
    else:
        a_gf = np.nan
        a_ga = np.nan
        a_form = np.nan

    home_goals_for.append(h_gf)
    home_goals_against.append(h_ga)

    away_goals_for.append(a_gf)
    away_goals_against.append(a_ga)

    home_form.append(h_form)
    away_form.append(a_form)

    if row["home_score"] > row["away_score"]:
        hp, ap = 3, 0
    elif row["home_score"] == row["away_score"]:
        hp, ap = 1, 1
    else:
        hp, ap = 0, 3

    team_history[home].append({
        "gf": row["home_score"],
        "ga": row["away_score"],
        "points": hp
    })

    team_history[away].append({
        "gf": row["away_score"],
        "ga": row["home_score"],
        "points": ap
    })

In [20]:
# Variables rolling

df["home_goals_for_6"] = home_goals_for
df["home_goals_against_6"] = home_goals_against

df["away_goals_for_6"] = away_goals_for
df["away_goals_against_6"] = away_goals_against

df["home_form_6"] = home_form
df["away_form_6"] = away_form

df["form_diff"] = (
    df["home_form_6"] -
    df["away_form_6"]
)

df[
    [
        "date",
        "home_team",
        "away_team",
        "home_goals_for_6",
        "home_goals_against_6",
        "away_goals_for_6",
        "away_goals_against_6",
        "home_form_6",
        "away_form_6",
        "form_diff"
    ]
].head(15)

,date,home_team,away_team,home_goals_for_6,home_goals_against_6,away_goals_for_6,away_goals_against_6,home_form_6,away_form_6,form_diff
0,1872-11-30,Scotland,England,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1873-03-08,England,Scotland,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
2,1874-03-07,Scotland,England,1.000000,2.000000,2.000000,1.000000,0.500000,2.000000,-1.500000
3,1875-03-06,England,Scotland,1.666667,1.333333,1.333333,1.666667,1.333333,1.333333,0.000000
4,1876-03-04,Scotland,England,1.500000,1.750000,1.750000,1.500000,1.250000,1.250000,0.000000
5,1876-03-25,Scotland,Wales,1.800000,1.400000,NaN,NaN,1.600000,NaN,NaN
6,1877-03-03,England,Scotland,1.400000,1.800000,2.166667,1.166667,1.000000,1.833333,-0.833333
7,1877-03-05,Wales,Scotland,0.000000,4.000000,2.666667,1.333333,0.000000,2.166667,-2.166667
8,1878-03-02,Scotland,England,2.666667,0.666667,1.333333,2.000000,2.666667,0.833333,1.833333
9,1878-03-23,Scotland,Wales,3.500000,0.833333,0.000000,3.000000,2.666667,0.000000,2.666667


## 11. Dataset con seis lags completos

In [21]:
# Dataset con seis lags completos

df_model = df.dropna(
    subset=elo_lag_cols
).copy()

print("df_model:", df_model.shape)
print(df_model["outcome"].value_counts())

df_model: (48878, 30)
outcome
H    23897
A    13800
D    11181
Name: count, dtype: int64


In [22]:
# Dataset con seis lags completos

X = df_model[elo_lag_cols].to_numpy(dtype=float)
y = df_model["outcome"].to_numpy()

print("X:", X.shape)
print("y:", y.shape)

X: (48878, 6)
y: (48878,)


# 12. Whitening

El artículo define:

`x_tilde = Sigma_X^(-1/2) (x - x_bar)`

En el backtest final, la media y covarianza deberán estimarse solamente con el entrenamiento anterior a la barrera temporal.

In [23]:
# Whitening

def whiten(X):

    X = np.asarray(X, dtype=float)

    X_mean = X.mean(axis=0)
    X_centered = X - X_mean

    X_cov = np.cov(
        X_centered,
        rowvar=False
    )

    eigenvalues, eigenvectors = np.linalg.eigh(X_cov)

    eigenvalues = np.maximum(
        eigenvalues,
        1e-10
    )

    inv_sqrt = (
        eigenvectors
        @ np.diag(1 / np.sqrt(eigenvalues))
        @ eigenvectors.T
    )

    X_white = X_centered @ inv_sqrt

    return X_white, X_mean, X_cov


X_white, X_mean, X_cov = whiten(X)

print(X_white.shape)

(48878, 6)


## 13. SIR

In [24]:
# SIR

def sir_kernel(X_white, y):

    X_white = np.asarray(X_white)
    y = np.asarray(y)

    n = len(y)
    K = X_white.shape[1]

    M = np.zeros((K, K))

    for c in np.unique(y):

        Xc = X_white[y == c]
        pi_c = len(Xc) / n
        mean_c = Xc.mean(axis=0)

        M += (
            pi_c
            * np.outer(mean_c, mean_c)
        )

    return M

## 14. SAVE

In [25]:
# SAVE

def save_kernel(X_white, y):

    X_white = np.asarray(X_white)
    y = np.asarray(y)

    n = len(y)
    K = X_white.shape[1]

    I = np.eye(K)
    M = np.zeros((K, K))

    for c in np.unique(y):

        Xc = X_white[y == c]

        pi_c = len(Xc) / n
        mean_c = Xc.mean(axis=0)

        Xc_centered = Xc - mean_c

        Sigma_c = np.cov(
            Xc_centered,
            rowvar=False
        )

        A = I - Sigma_c

        M += (
            pi_c
            * A
            @ A
        )

    return M

## 15. Direcciones SDR

In [26]:
# Direcciones SDR

def get_sdr_directions(M, d):

    eigenvalues, eigenvectors = np.linalg.eigh(M)

    idx = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    B = eigenvectors[:, :d]

    return eigenvalues, B


def project_sdr(X_white, B):
    return X_white @ B

## 16. SIR d=1 y d=2

In [27]:
# SIR d=1 y d=2

M_sir = sir_kernel(X_white, y)

eigenvalues_sir_1, B_sir_1 = get_sdr_directions(
    M_sir, d=1
)

eigenvalues_sir_2, B_sir_2 = get_sdr_directions(
    M_sir, d=2
)

Z_sir_1 = project_sdr(X_white, B_sir_1)
Z_sir_2 = project_sdr(X_white, B_sir_2)

print("M_sir:", M_sir.shape)
print("B_sir_1:", B_sir_1.shape)
print("B_sir_2:", B_sir_2.shape)
print("Z_sir_1:", Z_sir_1.shape)
print("Z_sir_2:", Z_sir_2.shape)

M_sir: (6, 6)
B_sir_1: (6, 1)
B_sir_2: (6, 2)
Z_sir_1: (48878, 1)
Z_sir_2: (48878, 2)


## 17. SAVE d=1 y d=2

In [28]:
# SAVE d=1 y d=2

M_save = save_kernel(X_white, y)

eigenvalues_save_1, B_save_1 = get_sdr_directions(
    M_save, d=1
)

eigenvalues_save_2, B_save_2 = get_sdr_directions(
    M_save, d=2
)

Z_save_1 = project_sdr(X_white, B_save_1)
Z_save_2 = project_sdr(X_white, B_save_2)

print("M_save:", M_save.shape)
print("B_save_1:", B_save_1.shape)
print("B_save_2:", B_save_2.shape)
print("Z_save_1:", Z_save_1.shape)
print("Z_save_2:", Z_save_2.shape)

M_save: (6, 6)
B_save_1: (6, 1)
B_save_2: (6, 2)
Z_save_1: (48878, 1)
Z_save_2: (48878, 2)


In [29]:
# SAVE d=1 y d=2

df_model["sir_1"] = Z_sir_1[:, 0]

df_model["sir_2_1"] = Z_sir_2[:, 0]
df_model["sir_2_2"] = Z_sir_2[:, 1]

df_model["save_1"] = Z_save_1[:, 0]

df_model["save_2_1"] = Z_save_2[:, 0]
df_model["save_2_2"] = Z_save_2[:, 1]

# 18. Poisson double regression

Home:
`log(lambda_H) = mu_H + sum(xi_j z_j) + delta_H N + eta1 G_h+ + eta2 G_a-`

Away:
`log(lambda_A) = mu_A - sum(xi_j z_j) + delta_A N + eta3 G_a+ + eta4 G_h-`

In [30]:
# Poisson double regression

def fit_poisson_double(
    data,
    score_cols,
    neutral_col="neutral"
):

    required = (
        score_cols
        + [
            neutral_col,
            "home_goals_for_6",
            "home_goals_against_6",
            "away_goals_for_6",
            "away_goals_against_6",
            "home_score",
            "away_score"
        ]
    )

    d = data.dropna(
        subset=required
    ).copy()


    X_home = d[
        score_cols
        + [
            neutral_col,
            "home_goals_for_6",
            "away_goals_against_6"
        ]
    ].copy()

    X_home = sm.add_constant(
        X_home,
        has_constant="add"
    )

    model_home = sm.GLM(
        d["home_score"],
        X_home,
        family=sm.families.Poisson()
    ).fit()


    X_away = d[
        score_cols
        + [
            neutral_col,
            "away_goals_for_6",
            "home_goals_against_6"
        ]
    ].copy()

    for col in score_cols:
        X_away[col] = -X_away[col]

    X_away = sm.add_constant(
        X_away,
        has_constant="add"
    )

    model_away = sm.GLM(
        d["away_score"],
        X_away,
        family=sm.families.Poisson()
    ).fit()

    return model_home, model_away, d

## 19. Ajustar M8–M11

In [31]:
# Ajustar M8–M11

models = {}

models["M8_SIR_d1"] = fit_poisson_double(
    df_model,
    ["sir_1"]
)

models["M9_SIR_d2"] = fit_poisson_double(
    df_model,
    ["sir_2_1", "sir_2_2"]
)

models["M10_SAVE_d1"] = fit_poisson_double(
    df_model,
    ["save_1"]
)

models["M11_SAVE_d2"] = fit_poisson_double(
    df_model,
    ["save_2_1", "save_2_2"]
)

print(list(models.keys()))

['M8_SIR_d1', 'M9_SIR_d2', 'M10_SAVE_d1', 'M11_SAVE_d2']


## 20. Probabilidades de marcador

In [32]:
# Probabilidades de marcador

def scoreline_probabilities(
    lambda_home,
    lambda_away,
    gmax=8
):

    p_home = 0.0
    p_draw = 0.0
    p_away = 0.0

    for gh in range(gmax + 1):

        for ga in range(gmax + 1):

            p = (
                poisson.pmf(gh, lambda_home)
                *
                poisson.pmf(ga, lambda_away)
            )

            if gh > ga:
                p_home += p
            elif gh == ga:
                p_draw += p
            else:
                p_away += p

    total = p_home + p_draw + p_away

    return np.array([
        p_home / total,
        p_draw / total,
        p_away / total
    ])

## 21. Predicción de un partido

In [33]:
# Predicción de un partido

def predict_match(
    model_home,
    model_away,
    match_row,
    score_cols
):

    Xh = pd.DataFrame({
        col: [match_row[col]]
        for col in score_cols
    })

    Xh["neutral"] = match_row["neutral"]
    Xh["home_goals_for_6"] = match_row["home_goals_for_6"]
    Xh["away_goals_against_6"] = match_row["away_goals_against_6"]

    Xh = sm.add_constant(
        Xh,
        has_constant="add"
    )

    lambda_home = float(
        model_home.predict(Xh)[0]
    )

    Xa = pd.DataFrame({
        col: [-match_row[col]]
        for col in score_cols
    })

    Xa["neutral"] = match_row["neutral"]
    Xa["away_goals_for_6"] = match_row["away_goals_for_6"]
    Xa["home_goals_against_6"] = match_row["home_goals_against_6"]

    Xa = sm.add_constant(
        Xa,
        has_constant="add"
    )

    lambda_away = float(
        model_away.predict(Xa)[0]
    )

    probabilities = scoreline_probabilities(
        lambda_home,
        lambda_away,
        gmax=8
    )

    return lambda_home, lambda_away, probabilities

## 22. RPS

In [34]:
# RPS

def rps_score(probabilities, actual):

    if actual == "H":
        y1, y2 = 1, 1
    elif actual == "D":
        y1, y2 = 0, 1
    else:
        y1, y2 = 0, 0

    return 0.5 * (
        (probabilities[0] - y1) ** 2
        +
        (
            probabilities[0]
            + probabilities[1]
            - y2
        ) ** 2
    )

# 23. Evaluación diagnóstica IN-SAMPLE

Esta parte solamente comprueba que el pipeline funciona.


In [35]:
# Evaluación diagnóstica IN-SAMPLE

def evaluate_model(
    data,
    model_home,
    model_away,
    score_cols
):

    rps_values = []
    predictions = []

    for _, row in data.iterrows():

        try:
            _, _, probs = predict_match(
                model_home,
                model_away,
                row,
                score_cols
            )

            rps_values.append(
                rps_score(
                    probs,
                    row["outcome"]
                )
            )

            prediction = ["H", "D", "A"][np.argmax(probs)]

            predictions.append(
                prediction == row["outcome"]
            )

        except Exception:
            continue

    return {
        "RPS": np.mean(rps_values),
        "Accuracy": np.mean(predictions),
        "N": len(rps_values)
    }

In [36]:
# RECONSTRUIR LOS DATASETS DE BACKTEST 2018 Y 2022

def build_backtest_sets(df, barrier_date, world_cup_year, qualified_teams):

    barrier_date = pd.Timestamp(barrier_date)


    test = df[
        (df["tournament"] == "FIFA World Cup") &
        (df["date"].dt.year == world_cup_year)
    ].copy()




    train = df[
        (df["date"] >= pd.Timestamp("2010-01-01")) &
        (df["date"] < barrier_date) &
        (
            df["home_team"].isin(qualified_teams) |
            df["away_team"].isin(qualified_teams)
        )
    ].copy()

    elo_cols = [
        "elo_diff_m0",
        "elo_diff_m1",
        "elo_diff_m2",
        "elo_diff_m3",
        "elo_diff_m4",
        "elo_diff_m5"
    ]

    train = train.dropna(subset=elo_cols).copy()
    test = test.dropna(subset=elo_cols).copy()

    return train, test






qualified_2018 = [
    "Russia",
    "Saudi Arabia",
    "Egypt",
    "Uruguay",
    "Portugal",
    "Spain",
    "Morocco",
    "Iran",
    "France",
    "Australia",
    "Peru",
    "Denmark",
    "Argentina",
    "Iceland",
    "Croatia",
    "Nigeria",
    "Brazil",
    "Switzerland",
    "Costa Rica",
    "Serbia",
    "Germany",
    "Mexico",
    "Sweden",
    "South Korea",
    "Belgium",
    "Panama",
    "Tunisia",
    "England",
    "Poland",
    "Senegal",
    "Colombia",
    "Japan"
]






qualified_2022 = [
    "Qatar",
    "Ecuador",
    "Senegal",
    "Netherlands",
    "England",
    "Iran",
    "USA",
    "Wales",
    "Argentina",
    "Saudi Arabia",
    "Mexico",
    "Poland",
    "France",
    "Australia",
    "Denmark",
    "Tunisia",
    "Spain",
    "Costa Rica",
    "Germany",
    "Japan",
    "Belgium",
    "Canada",
    "Morocco",
    "Croatia",
    "Brazil",
    "Serbia",
    "Switzerland",
    "Cameroon",
    "Portugal",
    "Ghana",
    "Uruguay",
    "South Korea"
]






train_2018, test_2018 = build_backtest_sets(
    df_model,
    "2018-06-14",
    2018,
    qualified_2018
)

train_2022, test_2022 = build_backtest_sets(
    df_model,
    "2022-11-20",
    2022,
    qualified_2022
)


print("2018")
print("Train:", train_2018.shape)
print("Test :", test_2018.shape)

print("\n2022")
print("Train:", train_2022.shape)
print("Test :", test_2022.shape)

2018
Train: (2757, 36)
Test : (64, 36)

2022
Train: (4062, 36)
Test : (64, 36)


In [37]:
# CONSTRUIR LOS EQUIPOS CLASIFICADOS DIRECTAMENTE

qualified_2018 = set(
    test_2018["home_team"].tolist() +
    test_2018["away_team"].tolist()
)



qualified_2022 = set(
    test_2022["home_team"].tolist() +
    test_2022["away_team"].tolist()
)

print("Equipos 2018:", len(qualified_2018))
print("Equipos 2022:", len(qualified_2022))






train_2018, test_2018 = build_backtest_sets(
    df_model,
    "2018-06-14",
    2018,
    qualified_2018
)

train_2022, test_2022 = build_backtest_sets(
    df_model,
    "2022-11-20",
    2022,
    qualified_2022
)






print("\n2018")
print("Train:", train_2018.shape)
print("Test :", test_2018.shape)

print("\n2022")
print("Train:", train_2022.shape)
print("Test :", test_2022.shape)

Equipos 2018: 32
Equipos 2022: 32

2018
Train: (2757, 36)
Test : (64, 36)

2022
Train: (4180, 36)
Test : (64, 36)


In [38]:
# EVALUACION DIAGNOSTICA

print("Pipeline completo hasta Poisson.")
print("Evaluación diagnóstica omitida.")
print("Siguiente paso: backtesting 2018 y 2022.")

Pipeline completo hasta Poisson.
Evaluación diagnóstica omitida.
Siguiente paso: backtesting 2018 y 2022.


In [39]:
# Equipos clasificados

WC_2018_TEAMS = set(
    df.loc[
        df["tournament"].eq("FIFA World Cup")
        & df["date"].dt.year.eq(2018),
        ["home_team", "away_team"]
    ].stack()
)

WC_2022_TEAMS = set(
    df.loc[
        df["tournament"].eq("FIFA World Cup")
        & df["date"].dt.year.eq(2022),
        ["home_team", "away_team"]
    ].stack()
)

print("Equipos Mundial 2018:", len(WC_2018_TEAMS))
print(sorted(WC_2018_TEAMS))

print("\nEquipos Mundial 2022:", len(WC_2022_TEAMS))
print(sorted(WC_2022_TEAMS))

Equipos Mundial 2018: 32
['Argentina', 'Australia', 'Belgium', 'Brazil', 'Colombia', 'Costa Rica', 'Croatia', 'Denmark', 'Egypt', 'England', 'France', 'Germany', 'Iceland', 'Iran', 'Japan', 'Mexico', 'Morocco', 'Nigeria', 'Panama', 'Peru', 'Poland', 'Portugal', 'Russia', 'Saudi Arabia', 'Senegal', 'Serbia', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Uruguay']

Equipos Mundial 2022: 32
['Argentina', 'Australia', 'Belgium', 'Brazil', 'Cameroon', 'Canada', 'Costa Rica', 'Croatia', 'Denmark', 'Ecuador', 'England', 'France', 'Germany', 'Ghana', 'Iran', 'Japan', 'Mexico', 'Morocco', 'Netherlands', 'Poland', 'Portugal', 'Qatar', 'Saudi Arabia', 'Senegal', 'Serbia', 'South Korea', 'Spain', 'Switzerland', 'Tunisia', 'United States', 'Uruguay', 'Wales']


In [40]:
# Funciones de entrenamiento

def build_backtest_sets(
    df,
    barrier_date,
    world_cup_year,
    qualified_teams
):

    barrier_date = pd.Timestamp(barrier_date)




    test = df[
        (df["tournament"] == "FIFA World Cup")
        & (df["date"].dt.year == world_cup_year)
    ].copy()






    train = df[
        (df["date"] >= pd.Timestamp("2010-01-01"))
        & (df["date"] < barrier_date)
        &
        (
            df["home_team"].isin(qualified_teams)
            |
            df["away_team"].isin(qualified_teams)
        )
    ].copy()


    elo_cols = [
        "elo_diff_m0",
        "elo_diff_m1",
        "elo_diff_m2",
        "elo_diff_m3",
        "elo_diff_m4",
        "elo_diff_m5",
    ]

    train = train.dropna(subset=elo_cols).copy()
    test = test.dropna(subset=elo_cols).copy()

    return train, test

In [41]:
# Creacion de los brackets

train_2018, test_2018 = build_backtest_sets(
    df=df,
    barrier_date="2018-06-14",
    world_cup_year=2018,
    qualified_teams=WC_2018_TEAMS
)

print("TRAIN 2018:", train_2018.shape)
print("TEST 2018:", test_2018.shape)

TRAIN 2018: (2757, 30)
TEST 2018: (64, 30)


In [42]:
# BACKTEST 2022

train_2022, test_2022 = build_backtest_sets(
    df=df,
    barrier_date="2022-11-20",
    world_cup_year=2022,
    qualified_teams=WC_2022_TEAMS
)

print("TRAIN 2022:", train_2022.shape)
print("TEST 2022:", test_2022.shape)

TRAIN 2022: (4180, 30)
TEST 2022: (64, 30)


In [43]:
# COMPROBACIÓN DE LOS PARTIDOS DE ENTRENAMIENTO

print("TRAIN 2018")
print(
    train_2018[
        [
            "date",
            "home_team",
            "away_team",
            "tournament"
        ]
    ].tail(10)
)

print("\n" + "=" * 70)

print("TRAIN 2022")
print(
    train_2022[
        [
            "date",
            "home_team",
            "away_team",
            "tournament"
        ]
    ].tail(10)
)

TRAIN 2018
            date    home_team   away_team tournament
41630 2018-06-09      Estonia     Morocco   Friendly
41631 2018-06-09       Sweden        Peru   Friendly
41632 2018-06-09      Denmark      Mexico   Friendly
41633 2018-06-09       Serbia     Bolivia   Friendly
41635 2018-06-09      Tunisia       Spain   Friendly
41636 2018-06-10      Austria      Brazil   Friendly
41638 2018-06-11      Belgium  Costa Rica   Friendly
41639 2018-06-11  South Korea     Senegal   Friendly
41640 2018-06-12        Japan    Paraguay   Friendly
41641 2018-06-12       Poland   Lithuania   Friendly

TRAIN 2022
            date    home_team away_team tournament
45656 2022-11-16       Mexico    Sweden   Friendly
45662 2022-11-16         Iran   Tunisia   Friendly
45667 2022-11-17  Switzerland     Ghana   Friendly
45674 2022-11-17     Portugal   Nigeria   Friendly
45675 2022-11-17      Morocco   Georgia   Friendly
45679 2022-11-17        Japan    Canada   Friendly
45684 2022-11-17       Jordan     Spa

In [44]:
# WHITENING PARA BACKTEST TEMPORAL

def fit_whitening(X_train):

    X_train = np.asarray(
        X_train,
        dtype=float
    )


    mean_train = X_train.mean(axis=0)


    X_centered = X_train - mean_train


    cov_train = np.cov(
        X_centered,
        rowvar=False
    )


    eigenvalues, eigenvectors = np.linalg.eigh(
        cov_train
    )


    eigenvalues = np.maximum(
        eigenvalues,
        1e-10
    )


    inv_sqrt = (
        eigenvectors
        @ np.diag(
            1 / np.sqrt(eigenvalues)
        )
        @ eigenvectors.T
    )

    return (
        mean_train,
        cov_train,
        inv_sqrt
    )


def transform_whitening(
    X,
    mean_train,
    inv_sqrt
):

    X = np.asarray(
        X,
        dtype=float
    )

    X_centered = X - mean_train

    return X_centered @ inv_sqrt

In [45]:
# SIR

def fit_sir(
    X_train_white,
    y_train,
    d
):

    X_train_white = np.asarray(
        X_train_white,
        dtype=float
    )

    y_train = np.asarray(
        y_train
    )

    n = len(y_train)
    K = X_train_white.shape[1]

    M = np.zeros(
        (K, K)
    )


    for c in ["H", "D", "A"]:

        Xc = X_train_white[
            y_train == c
        ]


        if len(Xc) == 0:
            continue

        pi_c = len(Xc) / n

        mean_c = Xc.mean(
            axis=0
        )

        M += (
            pi_c
            *
            np.outer(
                mean_c,
                mean_c
            )
        )


    eigenvalues, eigenvectors = np.linalg.eigh(M)


    idx = np.argsort(
        eigenvalues
    )[::-1]

    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]


    B = eigenvectors[:, :d]

    return (
        B,
        eigenvalues
    )

In [46]:
# SAVE

def fit_save(
    X_train_white,
    y_train,
    d
):

    X_train_white = np.asarray(
        X_train_white,
        dtype=float
    )

    y_train = np.asarray(
        y_train
    )

    n = len(y_train)
    K = X_train_white.shape[1]

    I = np.eye(K)

    M = np.zeros(
        (K, K)
    )

    for c in ["H", "D", "A"]:

        Xc = X_train_white[
            y_train == c
        ]

        if len(Xc) < 2:
            continue

        pi_c = len(Xc) / n


        mean_c = Xc.mean(
            axis=0
        )


        Xc_centered = (
            Xc - mean_c
        )


        Sigma_c = np.cov(
            Xc_centered,
            rowvar=False
        )


        A = I - Sigma_c


        M += (
            pi_c
            * (A @ A)
        )


    eigenvalues, eigenvectors = np.linalg.eigh(M)

    idx = np.argsort(
        eigenvalues
    )[::-1]

    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    B = eigenvectors[:, :d]

    return (
        B,
        eigenvalues
    )

In [47]:
# DATOS SDR — BACKTEST 2018

X_train_2018 = train_2018[
    elo_lag_cols
].to_numpy(
    dtype=float
)

y_train_2018 = train_2018[
    "outcome"
].to_numpy()

X_test_2018 = test_2018[
    elo_lag_cols
].to_numpy(
    dtype=float
)

y_test_2018 = test_2018[
    "outcome"
].to_numpy()

print(
    "X_train_2018:",
    X_train_2018.shape
)

print(
    "X_test_2018:",
    X_test_2018.shape
)

print(
    "\ny_train:",
    len(y_train_2018)
)

print(
    "y_test:",
    len(y_test_2018)
)

X_train_2018: (2757, 6)
X_test_2018: (64, 6)

y_train: 2757
y_test: 64


In [48]:
# WHITENING — 2018

mean_2018, cov_2018, inv_sqrt_2018 = fit_whitening(
    X_train_2018
)

X_train_white_2018 = transform_whitening(
    X_train_2018,
    mean_2018,
    inv_sqrt_2018
)

X_test_white_2018 = transform_whitening(
    X_test_2018,
    mean_2018,
    inv_sqrt_2018
)

print(
    "TRAIN whitened:",
    X_train_white_2018.shape
)

print(
    "TEST whitened:",
    X_test_white_2018.shape
)

TRAIN whitened: (2757, 6)
TEST whitened: (64, 6)


In [49]:
# WHITENING — 2018

mean_2018, cov_2018, inv_sqrt_2018 = fit_whitening(
    X_train_2018
)

X_train_white_2018 = transform_whitening(
    X_train_2018,
    mean_2018,
    inv_sqrt_2018
)

X_test_white_2018 = transform_whitening(
    X_test_2018,
    mean_2018,
    inv_sqrt_2018
)

print(
    "TRAIN whitened:",
    X_train_white_2018.shape
)

print(
    "TEST whitened:",
    X_test_white_2018.shape
)

TRAIN whitened: (2757, 6)
TEST whitened: (64, 6)


In [50]:
# SIR — M8 Y M9 — 2018

B_sir_2018_d1, eig_sir_2018 = fit_sir(
    X_train_white_2018,
    y_train_2018,
    d=1
)


B_sir_2018_d2, _ = fit_sir(
    X_train_white_2018,
    y_train_2018,
    d=2
)


Z_sir_2018_d1_train = project_sdr(
    X_train_white_2018,
    B_sir_2018_d1
)

Z_sir_2018_d2_train = project_sdr(
    X_train_white_2018,
    B_sir_2018_d2
)


Z_sir_2018_d1_test = project_sdr(
    X_test_white_2018,
    B_sir_2018_d1
)

Z_sir_2018_d2_test = project_sdr(
    X_test_white_2018,
    B_sir_2018_d2
)

print(
    "M8 TRAIN:",
    Z_sir_2018_d1_train.shape
)

print(
    "M8 TEST:",
    Z_sir_2018_d1_test.shape
)

print(
    "M9 TRAIN:",
    Z_sir_2018_d2_train.shape
)

print(
    "M9 TEST:",
    Z_sir_2018_d2_test.shape
)

M8 TRAIN: (2757, 1)
M8 TEST: (64, 1)
M9 TRAIN: (2757, 2)
M9 TEST: (64, 2)


In [51]:
# SAVE — M10 Y M11 — 2018

B_save_2018_d1, eig_save_2018 = fit_save(
    X_train_white_2018,
    y_train_2018,
    d=1
)


B_save_2018_d2, _ = fit_save(
    X_train_white_2018,
    y_train_2018,
    d=2
)


Z_save_2018_d1_train = project_sdr(
    X_train_white_2018,
    B_save_2018_d1
)

Z_save_2018_d2_train = project_sdr(
    X_train_white_2018,
    B_save_2018_d2
)


Z_save_2018_d1_test = project_sdr(
    X_test_white_2018,
    B_save_2018_d1
)

Z_save_2018_d2_test = project_sdr(
    X_test_white_2018,
    B_save_2018_d2
)

print(
    "M10 TRAIN:",
    Z_save_2018_d1_train.shape
)

print(
    "M10 TEST:",
    Z_save_2018_d1_test.shape
)

print(
    "M11 TRAIN:",
    Z_save_2018_d2_train.shape
)

print(
    "M11 TEST:",
    Z_save_2018_d2_test.shape
)

M10 TRAIN: (2757, 1)
M10 TEST: (64, 1)
M11 TRAIN: (2757, 2)
M11 TEST: (64, 2)


In [52]:
# VARIABLES SDR — 2018

train_2018_sdr = train_2018.copy()
test_2018_sdr = test_2018.copy()





train_2018_sdr["sir_1"] = (
    Z_sir_2018_d1_train[:, 0]
)

test_2018_sdr["sir_1"] = (
    Z_sir_2018_d1_test[:, 0]
)






train_2018_sdr["sir_2_1"] = (
    Z_sir_2018_d2_train[:, 0]
)

train_2018_sdr["sir_2_2"] = (
    Z_sir_2018_d2_train[:, 1]
)

test_2018_sdr["sir_2_1"] = (
    Z_sir_2018_d2_test[:, 0]
)

test_2018_sdr["sir_2_2"] = (
    Z_sir_2018_d2_test[:, 1]
)






train_2018_sdr["save_1"] = (
    Z_save_2018_d1_train[:, 0]
)

test_2018_sdr["save_1"] = (
    Z_save_2018_d1_test[:, 0]
)






train_2018_sdr["save_2_1"] = (
    Z_save_2018_d2_train[:, 0]
)

train_2018_sdr["save_2_2"] = (
    Z_save_2018_d2_train[:, 1]
)

test_2018_sdr["save_2_1"] = (
    Z_save_2018_d2_test[:, 0]
)

test_2018_sdr["save_2_2"] = (
    Z_save_2018_d2_test[:, 1]
)


print(
    train_2018_sdr[
        [
            "sir_1",
            "sir_2_1",
            "sir_2_2",
            "save_1",
            "save_2_1",
            "save_2_2"
        ]
    ].head()
)

          sir_1   sir_2_1   sir_2_2    save_1  save_2_1  save_2_2
33591 -0.295426 -0.295426 -1.093801 -0.126767 -0.126767 -0.221031
33596 -0.316955 -0.316955  0.621197 -0.261598 -0.261598 -0.289612
33599  1.928435  1.928435  0.095306  1.891945  1.891945 -0.208757
33602  0.963332  0.963332 -0.508352  1.077744  1.077744 -0.340902
33609  1.237378  1.237378 -0.474937  1.330903  1.330903  0.026404


In [53]:
# POISSON DOUBLE REGRESSION — BACKTEST 2018

def fit_poisson_sdr(
    train_data,
    score_cols
):

    required_cols = (
        score_cols
        + [
            "neutral",
            "home_goals_for_6",
            "home_goals_against_6",
            "away_goals_for_6",
            "away_goals_against_6",
            "home_score",
            "away_score"
        ]
    )

    data = train_data.dropna(
        subset=required_cols
    ).copy()





    X_home = data[
        score_cols
        + [
            "neutral",
            "home_goals_for_6",
            "away_goals_against_6"
        ]
    ].copy()

    X_home = sm.add_constant(
        X_home,
        has_constant="add"
    )

    model_home = sm.GLM(
        data["home_score"],
        X_home,
        family=sm.families.Poisson()
    ).fit()





    X_away = data[
        score_cols
        + [
            "neutral",
            "away_goals_for_6",
            "home_goals_against_6"
        ]
    ].copy()


    for col in score_cols:
        X_away[col] = -X_away[col]

    X_away = sm.add_constant(
        X_away,
        has_constant="add"
    )

    model_away = sm.GLM(
        data["away_score"],
        X_away,
        family=sm.families.Poisson()
    ).fit()

    return model_home, model_away, data

In [54]:
# AJUSTAR M8, M9, M10 Y M11 — 2018

poisson_2018 = {}





poisson_2018["M8"] = fit_poisson_sdr(
    train_2018_sdr,
    ["sir_1"]
)






poisson_2018["M9"] = fit_poisson_sdr(
    train_2018_sdr,
    ["sir_2_1", "sir_2_2"]
)






poisson_2018["M10"] = fit_poisson_sdr(
    train_2018_sdr,
    ["save_1"]
)






poisson_2018["M11"] = fit_poisson_sdr(
    train_2018_sdr,
    ["save_2_1", "save_2_2"]
)


print("Modelos Poisson ajustados:")
print(list(poisson_2018.keys()))

Modelos Poisson ajustados:
['M8', 'M9', 'M10', 'M11']


In [55]:
# COEFICIENTES POISSON — 2018

for name, (model_home, model_away, data_used) in poisson_2018.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("\nHOME")
    print(model_home.params)

    print("\nAWAY")
    print(model_away.params)


M8

HOME
const                   0.221612
sir_1                  -0.449662
neutral                -0.156924
home_goals_for_6        0.050287
away_goals_against_6    0.060511
dtype: float64

AWAY
const                  -0.325531
sir_1                  -0.435030
neutral                 0.158127
away_goals_for_6        0.098179
home_goals_against_6    0.101014
dtype: float64

M9

HOME
const                   0.222594
sir_2_1                -0.449702
sir_2_2                 0.012436
neutral                -0.156709
home_goals_for_6        0.049919
away_goals_against_6    0.060074
dtype: float64

AWAY
const                  -0.328644
sir_2_1                -0.434631
sir_2_2                 0.025823
neutral                 0.161352
away_goals_for_6        0.099245
home_goals_against_6    0.101331
dtype: float64

M10

HOME
const                   0.202669
save_1                 -0.443454
neutral                -0.165708
home_goals_for_6        0.056879
away_goals_against_6    0.071329
dtype:

In [56]:
# PREDICCION DE UN PARTIDO

def predict_poisson_sdr(
    model_home,
    model_away,
    row,
    score_cols
):







    if isinstance(row, pd.Series):

        X_home_base = pd.DataFrame({
            col: [float(row[col])] for col in score_cols
        })
        X_home_base["neutral"] = float(row["neutral"])
        X_home_base["home_goals_for_6"] = float(row["home_goals_for_6"])
        X_home_base["away_goals_against_6"] = float(row["away_goals_against_6"])
    else:

        X_home_base = row[
            score_cols
            + ["neutral", "home_goals_for_6", "away_goals_against_6"]
        ].astype(float)



    X_home_with_const_df = sm.add_constant(
        X_home_base,
        has_constant="add"
    )


    X_home_predict_array = X_home_with_const_df.values


    prediction_result_home = model_home.predict(X_home_predict_array)


    lambda_home = float(prediction_result_home[0])






    if isinstance(row, pd.Series):
        X_away_base = pd.DataFrame({
            col: [-float(row[col])] for col in score_cols
        })
        X_away_base["neutral"] = float(row["neutral"])
        X_away_base["away_goals_for_6"] = float(row["away_goals_for_6"])
        X_away_base["home_goals_against_6"] = float(row["home_goals_against_6"])
    else:
        X_away_base = row[
            score_cols
            + ["neutral", "away_goals_for_6", "home_goals_against_6"]
        ].copy()
        for col in score_cols:
            X_away_base[col] = -X_away_base[col]
        X_away_base = X_away_base.astype(float)


    X_away_with_const_df = sm.add_constant(
        X_away_base,
        has_constant="add"
    )
    X_away_predict_array = X_away_with_const_df.values

    prediction_result_away = model_away.predict(X_away_predict_array)
    lambda_away = float(prediction_result_away[0])


    return lambda_home, lambda_away

In [57]:
# PROBABILIDADES H / D / A

def get_match_probabilities(
    lambda_home,
    lambda_away,
    gmax=8
):

    goals = np.arange(
        gmax + 1
    )

    p_home_goals = poisson.pmf(
        goals,
        lambda_home
    )

    p_away_goals = poisson.pmf(
        goals,
        lambda_away
    )




    joint = np.outer(
        p_home_goals,
        p_away_goals
    )

    p_home = np.tril(
        joint,
        k=-1
    ).sum()

    p_draw = np.trace(
        joint
    )

    p_away = np.triu(
        joint,
        k=1
    ).sum()

    total = (
        p_home
        + p_draw
        + p_away
    )

    return np.array([
        p_home / total,
        p_draw / total,
        p_away / total
    ])

In [58]:
# PREDICCIONES DEL MUNDIAL 2018

model_score_cols = {
    "M8": ["sir_1"],
    "M9": ["sir_2_1", "sir_2_2"],
    "M10": ["save_1"],
    "M11": ["save_2_1", "save_2_2"]
}


predictions_2018 = {}


for model_name, (
    model_home,
    model_away,
    _
) in poisson_2018.items():

    score_cols = model_score_cols[
        model_name
    ]

    rows = []

    for _, row in test_2018_sdr.iterrows():

        lambda_home, lambda_away = (
            predict_poisson_sdr(
                model_home,
                model_away,
                row,
                score_cols
            )
        )

        probabilities = (
            get_match_probabilities(
                lambda_home,
                lambda_away
            )
        )

        rows.append({
            "date": row["date"],
            "home_team": row["home_team"],
            "away_team": row["away_team"],
            "actual": row["outcome"],
            "lambda_home": lambda_home,
            "lambda_away": lambda_away,
            "p_home": probabilities[0],
            "p_draw": probabilities[1],
            "p_away": probabilities[2]
        })

    predictions_2018[model_name] = pd.DataFrame(
        rows
    )


print("Predicciones generadas.")

for model_name, pred in predictions_2018.items():
    print(
        model_name,
        "→",
        pred.shape
    )

Predicciones generadas.
M8 → (64, 9)
M9 → (64, 9)
M10 → (64, 9)
M11 → (64, 9)


In [59]:
# PRIMERAS PREDICCIONES — 2018

predictions_2018["M9"].head(10)

,date,home_team,away_team,actual,lambda_home,lambda_away,p_home,p_draw,p_away
0,2018-06-14,Russia,Saudi Arabia,H,1.620281,0.892865,0.544988,0.248082,0.206931
1,2018-06-15,Egypt,Uruguay,A,0.851586,1.547397,0.206716,0.256179,0.537105
2,2018-06-15,Morocco,Iran,A,1.053703,1.191640,0.322163,0.286895,0.390942
3,2018-06-15,Portugal,Spain,D,1.012137,1.520893,0.251996,0.257597,0.490407
4,2018-06-16,Croatia,Nigeria,H,1.550231,0.840927,0.540563,0.255847,0.203590
5,2018-06-16,France,Australia,H,1.691559,0.848665,0.573580,0.239778,0.186642
6,2018-06-16,Argentina,Iceland,D,2.106783,0.847225,0.664118,0.195990,0.139892
7,2018-06-16,Peru,Denmark,A,1.366225,0.859655,0.486744,0.277255,0.236001
8,2018-06-17,Brazil,Switzerland,D,1.780199,0.698737,0.633127,0.225409,0.141464
9,2018-06-17,Germany,Mexico,A,1.459912,0.824850,0.521117,0.266624,0.212259


In [60]:
# RPS

def calculate_rps(probabilities, actual):

    p_home = probabilities[0]
    p_draw = probabilities[1]

    if actual == "H":
        y_home = 1
        y_home_draw = 1

    elif actual == "D":
        y_home = 0
        y_home_draw = 1

    else:
        y_home = 0
        y_home_draw = 0

    return 0.5 * (
        (p_home - y_home) ** 2
        +
        (
            p_home
            + p_draw
            - y_home_draw
        ) ** 2
    )

In [61]:
# EVALUACION 2018

results_2018 = {}


for model_name, pred in predictions_2018.items():

    rps_values = []

    correct = 0

    for _, row in pred.iterrows():

        probabilities = np.array([
            row["p_home"],
            row["p_draw"],
            row["p_away"]
        ])


        rps = calculate_rps(
            probabilities,
            row["actual"]
        )

        rps_values.append(rps)


        predicted = ["H", "D", "A"][
            np.argmax(probabilities)
        ]

        if predicted == row["actual"]:
            correct += 1


    results_2018[model_name] = {
        "RPS": np.mean(rps_values),
        "Accuracy": correct / len(pred),
        "N": len(pred)
    }


results_2018_df = (
    pd.DataFrame(results_2018)
    .T
)

results_2018_df

,RPS,Accuracy,N
M8,0.209811,0.546875,64.0
M9,0.206971,0.562500,64.0
M10,0.222982,0.515625,64.0
M11,0.208214,0.531250,64.0


In [62]:
# PREPARACIÓN DEL BACKTEST 2022

elo_lag_cols = [
    "elo_diff_m0",
    "elo_diff_m1",
    "elo_diff_m2",
    "elo_diff_m3",
    "elo_diff_m4",
    "elo_diff_m5"
]





X_train_2022 = train_2022[elo_lag_cols].to_numpy(dtype=float)
y_train_2022 = train_2022["outcome"].to_numpy()

X_test_2022 = test_2022[elo_lag_cols].to_numpy(dtype=float)
y_test_2022 = test_2022["outcome"].to_numpy()

print("X_train:", X_train_2022.shape)
print("X_test :", X_test_2022.shape)
print("y_train:", y_train_2022.shape)
print("y_test :", y_test_2022.shape)

X_train: (4180, 6)
X_test : (64, 6)
y_train: (4180,)
y_test : (64,)


In [63]:
# WHITENING 2022

mean_2022, cov_2022, inv_sqrt_2022 = fit_whitening(
    X_train_2022
)

X_train_2022_white = transform_whitening(
    X_train_2022,
    mean_2022,
    inv_sqrt_2022
)

X_test_2022_white = transform_whitening(
    X_test_2022,
    mean_2022,
    inv_sqrt_2022
)

print("Train whitened:", X_train_2022_white.shape)
print("Test whitened :", X_test_2022_white.shape)

Train whitened: (4180, 6)
Test whitened : (64, 6)


In [64]:
# SIR 2022

B_sir_2022_d1, _ = fit_sir(
    X_train_2022_white,
    y_train_2022,
    d=1
)

B_sir_2022_d2, _ = fit_sir(
    X_train_2022_white,
    y_train_2022,
    d=2
)


Z_train_2022_sir_d1 = project_sdr(
    X_train_2022_white,
    B_sir_2022_d1
)

Z_test_2022_sir_d1 = project_sdr(
    X_test_2022_white,
    B_sir_2022_d1
)

Z_train_2022_sir_d2 = project_sdr(
    X_train_2022_white,
    B_sir_2022_d2
)

Z_test_2022_sir_d2 = project_sdr(
    X_test_2022_white,
    B_sir_2022_d2
)

print("SIR d=1")
print("Train:", Z_train_2022_sir_d1.shape)
print("Test :", Z_test_2022_sir_d1.shape)

print("\nSIR d=2")
print("Train:", Z_train_2022_sir_d2.shape)
print("Test :", Z_test_2022_sir_d2.shape)

SIR d=1
Train: (4180, 1)
Test : (64, 1)

SIR d=2
Train: (4180, 2)
Test : (64, 2)


In [65]:
# SAVE 2022

B_save_2022_d1, _ = fit_save(
    X_train_2022_white,
    y_train_2022,
    d=1
)

B_save_2022_d2, _ = fit_save(
    X_train_2022_white,
    y_train_2022,
    d=2
)


Z_train_2022_save_d1 = project_sdr(
    X_train_2022_white,
    B_save_2022_d1
)

Z_test_2022_save_d1 = project_sdr(
    X_test_2022_white,
    B_save_2022_d1
)

Z_train_2022_save_d2 = project_sdr(
    X_train_2022_white,
    B_save_2022_d2
)

Z_test_2022_save_d2 = project_sdr(
    X_test_2022_white,
    B_save_2022_d2
)

print("SAVE d=1")
print("Train:", Z_train_2022_save_d1.shape)
print("Test :", Z_test_2022_save_d1.shape)

print("\nSAVE d=2")
print("Train:", Z_train_2022_save_d2.shape)
print("Test :", Z_test_2022_save_d2.shape)

SAVE d=1
Train: (4180, 1)
Test : (64, 1)

SAVE d=2
Train: (4180, 2)
Test : (64, 2)


In [66]:
# DATASETS SDR 2022

train_2022_sdr = train_2022.copy()
test_2022_sdr = test_2022.copy()





train_2022_sdr["sir_1"] = Z_train_2022_sir_d1[:, 0]
test_2022_sdr["sir_1"] = Z_test_2022_sir_d1[:, 0]

train_2022_sdr["sir_2_1"] = Z_train_2022_sir_d2[:, 0]
train_2022_sdr["sir_2_2"] = Z_train_2022_sir_d2[:, 1]

test_2022_sdr["sir_2_1"] = Z_test_2022_sir_d2[:, 0]
test_2022_sdr["sir_2_2"] = Z_test_2022_sir_d2[:, 1]





train_2022_sdr["save_1"] = Z_train_2022_save_d1[:, 0]
test_2022_sdr["save_1"] = Z_test_2022_save_d1[:, 0]

train_2022_sdr["save_2_1"] = Z_train_2022_save_d2[:, 0]
train_2022_sdr["save_2_2"] = Z_train_2022_save_d2[:, 1]

test_2022_sdr["save_2_1"] = Z_test_2022_save_d2[:, 0]
test_2022_sdr["save_2_2"] = Z_test_2022_save_d2[:, 1]

print("Train SDR:", train_2022_sdr.shape)
print("Test SDR :", test_2022_sdr.shape)

display(
    test_2022_sdr[
        [
            "date",
            "home_team",
            "away_team",
            "sir_1",
            "sir_2_1",
            "sir_2_2",
            "save_1",
            "save_2_1",
            "save_2_2"
        ]
    ].head()
)

Train SDR: (4180, 36)
Test SDR : (64, 36)


,date,home_team,away_team,sir_1,sir_2_1,sir_2_2,save_1,save_2_1,save_2_2
45703,2022-11-20,Qatar,Ecuador,0.476217,0.476217,-0.499622,0.423182,0.423182,1.342882
45721,2022-11-21,Senegal,Netherlands,0.904746,0.904746,0.369742,0.848652,0.848652,0.020070
45722,2022-11-21,England,Iran,-0.338314,-0.338314,-0.547765,-0.394619,-0.394619,1.163686
45723,2022-11-21,United States,Wales,-0.146926,-0.146926,0.141170,-0.155625,-0.155625,-0.048042
45724,2022-11-22,France,Australia,-0.810072,-0.810072,0.667666,-0.865277,-0.865277,-0.133607


In [67]:
# AJUSTE DE MODELOS SDR - 2022

models_2022 = {}


models_2022["M8"] = fit_poisson_sdr(
    train_2022_sdr,
    ["sir_1"]
)


models_2022["M9"] = fit_poisson_sdr(
    train_2022_sdr,
    ["sir_2_1", "sir_2_2"]
)


models_2022["M10"] = fit_poisson_sdr(
    train_2022_sdr,
    ["save_1"]
)


models_2022["M11"] = fit_poisson_sdr(
    train_2022_sdr,
    ["save_2_1", "save_2_2"]
)

print("Modelos ajustados:")
for name, (model_home, model_away, data_used) in models_2022.items():
    print(name, "n =", len(data_used))

Modelos ajustados:
M8 n = 4180
M9 n = 4180
M10 n = 4180
M11 n = 4180


In [68]:
# A. INSPECCIONAR LA SALIDA DE LA FUNCIÓN

result = predict_poisson_sdr(
    models_2022["M8"][0],
    models_2022["M8"][1],
    test_2022_sdr,
    ["sir_1"]
)

print("Tipo de result:", type(result))

if isinstance(result, tuple):
    print("Número de elementos:", len(result))

    for i, item in enumerate(result):
        print(
            f"Elemento {i}: "
            f"tipo={type(item)}, "
            f"shape={getattr(item, 'shape', 'sin shape')}"
        )
else:
    print("Shape:", getattr(result, "shape", "sin shape"))

Tipo de result: <class 'tuple'>
Número de elementos: 2
Elemento 0: tipo=<class 'float'>, shape=sin shape
Elemento 1: tipo=<class 'float'>, shape=sin shape


In [69]:
# Evaluación diagnóstica IN-SAMPLE

from scipy.stats import poisson
import numpy as np
import pandas
import statsmodels.api as sm

def predict_match_poisson_sdr(
    model_home,
    model_away,
    row,
    score_cols,
    gmax=8
):



    home_cols = score_cols + [
        "neutral",
        "home_goals_for_6",
        "away_goals_against_6"
    ]


    X_home_df = pandas.DataFrame(
        [[row[col] for col in home_cols]],
        columns=home_cols
    )

    X_home_df = X_home_df.apply(pandas.to_numeric, errors="raise")
    X_home_df = sm.add_constant(X_home_df, has_constant="add")


    X_home_predict_array = X_home_df.values


    prediction_result_home = model_home.predict(X_home_predict_array)


    lambda_home = float(prediction_result_home[0])






    away_cols = score_cols + [
        "neutral",
        "away_goals_for_6",
        "home_goals_against_6"
    ]


    X_away_df = pandas.DataFrame(
        [[row[col] for col in away_cols]],
        columns=away_cols
    )

    X_away_df = X_away_df.apply(pandas.to_numeric, errors="raise")


    for col in score_cols:
        X_away_df[col] = -X_away_df[col]

    X_away_df = sm.add_constant(X_away_df, has_constant="add")


    X_away_predict_array = X_away_df.values




    lambda_away = float(model_away.predict(X_away_predict_array)[0])




    goals = np.arange(gmax + 1)

    p_home_goals = poisson.pmf(
        goals,
        lambda_home
    )

    p_away_goals = poisson.pmf(
        goals,
        lambda_away
    )

    score_matrix = np.outer(
        p_home_goals,
        p_away_goals
    )






    p_home = np.tril(
        score_matrix,
        -1
    ).sum()


    p_draw = np.trace(score_matrix)



    p_away = np.triu(
        score_matrix,
        1
    ).sum()


    total = p_home + p_draw + p_away

    p_home /= total
    p_draw /= total
    p_away /= total

    return p_home, p_draw, p_away

In [70]:
# C. PREDICCIONES PARA TODOS LOS PARTIDOS

def predict_dataset_poisson_sdr(
    model_home,
    model_away,
    data,
    score_cols
):

    predictions = []

    for _, row in data.iterrows():

        p_home, p_draw, p_away = predict_match_poisson_sdr(
            model_home,
            model_away,
            row,
            score_cols
        )

        predictions.append({
            "date": row["date"],
            "home_team": row["home_team"],
            "away_team": row["away_team"],
            "actual": row["outcome"],
            "p_home": p_home,
            "p_draw": p_draw,
            "p_away": p_away
        })

    return pd.DataFrame(predictions)

In [71]:
# F. PREDICCIONES 2022

predictions_2022 = {}

score_cols_2022 = {
    "M8": ["sir_1"],
    "M9": ["sir_2_1", "sir_2_2"],
    "M10": ["save_1"],
    "M11": ["save_2_1", "save_2_2"]
}

for model_name, (model_home, model_away, _) in models_2022.items():

    predictions_2022[model_name] = predict_dataset_poisson_sdr(
        model_home,
        model_away,
        test_2022_sdr,
        score_cols_2022[model_name]
    )

for model_name, pred in predictions_2022.items():
    print(model_name, pred.shape)

M8 (64, 7)
M9 (64, 7)
M10 (64, 7)
M11 (64, 7)


In [72]:
# EVALUACIÓN 2022

results_2022 = {}

for model_name, pred in predictions_2022.items():

    rps_values = []
    correct = 0

    for _, row in pred.iterrows():

        probabilities = np.array([
            row["p_home"],
            row["p_draw"],
            row["p_away"]
        ])


        rps = calculate_rps(
            probabilities,
            row["actual"]
        )

        rps_values.append(rps)


        predicted = ["H", "D", "A"][
            np.argmax(probabilities)
        ]

        if predicted == row["actual"]:
            correct += 1

    results_2022[model_name] = {
        "RPS": np.mean(rps_values),
        "Accuracy": correct / len(pred),
        "N": len(pred)
    }

results_2022_df = pd.DataFrame(
    results_2022
).T

results_2022_df

,RPS,Accuracy,N
M8,0.227409,0.484375,64.0
M9,0.227329,0.484375,64.0
M10,0.232004,0.484375,64.0
M11,0.228852,0.484375,64.0


In [73]:
# AUDITORÍA DE LOS DATOS 2022

print("TRAIN 2022")
print("N:", len(train_2022))

print("\nDistribución de outcomes:")
print(train_2022["outcome"].value_counts())

print("\nProporciones:")
print(train_2022["outcome"].value_counts(normalize=True))

print("\nMedia Elo diff actual por outcome:")
print(
    train_2022
    .groupby("outcome")["elo_diff_m0"]
    .mean()
)

print("\nMedia de los 6 Elo lags:")
print(
    train_2022[
        [
            "elo_diff_m0",
            "elo_diff_m1",
            "elo_diff_m2",
            "elo_diff_m3",
            "elo_diff_m4",
            "elo_diff_m5"
        ]
    ].describe()
)

TRAIN 2022
N: 4180

Distribución de outcomes:
outcome
H    1990
A    1214
D     976
Name: count, dtype: int64

Proporciones:
outcome
H    0.476077
A    0.290431
D    0.233493
Name: proportion, dtype: float64

Media Elo diff actual por outcome:
outcome
A   -171.359709
D     -5.134965
H    163.629888
Name: elo_diff_m0, dtype: float64

Media de los 6 Elo lags:
       elo_diff_m0  elo_diff_m1  elo_diff_m2  elo_diff_m3  elo_diff_m4  \
count  4180.000000  4180.000000  4180.000000  4180.000000  4180.000000   
mean     26.933269    26.492442    26.205361    26.085112    26.625528   
std     263.680577   263.629709   263.771064   263.494531   263.276417   
min   -1092.412599 -1092.412599 -1092.412599 -1092.412599 -1086.907982   
25%    -145.472392  -145.036457  -146.150666  -146.292003  -145.647809   
50%      35.163838    35.107267    35.145567    35.904820    35.228756   
75%     198.250024   200.058845   197.462538   197.751046   197.831976   
max    1105.327414  1105.327414  1092.412599  10

In [74]:
# AUDITORÍA DE LOS DATOS 2018

print("TRAIN 2018")
print("N:", len(train_2018))

print("\nDistribución de outcomes:")
print(train_2018["outcome"].value_counts())

print("\nProporciones:")
print(train_2018["outcome"].value_counts(normalize=True))

print("\nMedia Elo diff actual por outcome:")
print(
    train_2018
    .groupby("outcome")["elo_diff_m0"]
    .mean()
)

TRAIN 2018
N: 2757

Distribución de outcomes:
outcome
H    1301
A     800
D     656
Name: count, dtype: int64

Proporciones:
outcome
H    0.47189
A    0.29017
D    0.23794
Name: proportion, dtype: float64

Media Elo diff actual por outcome:
outcome
A   -165.175320
D     -6.848698
H    150.009345
Name: elo_diff_m0, dtype: float64


In [75]:
# COEFICIENTES DE LOS MODELOS 2022

for model_name in ["M9", "M11"]:

    model_home = models_2022[model_name][0]
    model_away = models_2022[model_name][1]

    print("\n" + "="*50)
    print(model_name)

    print("\nHOME")
    print(model_home.params)

    print("\nAWAY")
    print(model_away.params)


M9

HOME
const                   0.238251
sir_2_1                -0.471682
sir_2_2                -0.001360
neutral                -0.176968
home_goals_for_6        0.051106
away_goals_against_6    0.062899
dtype: float64

AWAY
const                  -0.325556
sir_2_1                -0.460089
sir_2_2                 0.010608
neutral                 0.207785
away_goals_for_6        0.073101
home_goals_against_6    0.098147
dtype: float64

M11

HOME
const                   0.230508
save_2_1               -0.469804
save_2_2               -0.007524
neutral                -0.176258
home_goals_for_6        0.053692
away_goals_against_6    0.066090
dtype: float64

AWAY
const                  -0.337090
save_2_1               -0.456728
save_2_2               -0.021542
neutral                 0.207180
away_goals_for_6        0.077339
home_goals_against_6    0.103434
dtype: float64


In [76]:
# TASAS DE GOLES PROMEDIO

for model_name in ["M9", "M11"]:

    model_home = models_2022[model_name][0]
    model_away = models_2022[model_name][1]

    print("\n", model_name)

    print(
        "Home mean:",
        np.mean(model_home.predict())
    )

    print(
        "Away mean:",
        np.mean(model_away.predict())
    )


 M9
Home mean: 1.6007177033492803
Away mean: 1.0894736842105268

 M11
Home mean: 1.6007177033492823
Away mean: 1.0894736842105264


In [77]:
# REVISIÓN DE VARIABLES DE FORMA

form_cols = [
    "home_goals_for_6",
    "home_goals_against_6",
    "away_goals_for_6",
    "away_goals_against_6"
]

train_2022[form_cols].describe()

,home_goals_for_6,home_goals_against_6,away_goals_for_6,away_goals_against_6
count,4180.000000,4180.000000,4180.000000,4180.000000
mean,1.572823,1.050271,1.504625,1.098485
std,0.740525,0.600902,0.733303,0.594416
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.666667,1.000000,0.666667
50%,1.500000,1.000000,1.333333,1.000000
75%,2.000000,1.333333,1.875000,1.333333
max,6.166667,6.666667,5.666667,5.333333


In [78]:
# Evaluación diagnóstica IN-SAMPLE

results_2022_df

,RPS,Accuracy,N
M8,0.227409,0.484375,64.0
M9,0.227329,0.484375,64.0
M10,0.232004,0.484375,64.0
M11,0.228852,0.484375,64.0


###2026

In [79]:
# Evaluación diagnóstica IN-SAMPLE

groups_2026 = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Ivory Coast", "Curaçao", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

qualified_2026 = sorted(
    {team for teams in groups_2026.values() for team in teams}
)

print("Número de equipos:", len(qualified_2026))
print(qualified_2026)

Número de equipos: 48
['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curaçao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']


In [80]:
# Evaluación diagnóstica IN-SAMPLE

barrier_2026 = pd.Timestamp("2026-06-01")

train_2026 = df_model[
    (df_model["date"] >= pd.Timestamp("2010-01-01"))
    & (df_model["date"] < barrier_2026)
    & (
        df_model["home_team"].isin(qualified_2026)
        | df_model["away_team"].isin(qualified_2026)
    )
].copy()

print("N entrenamiento 2026:", len(train_2026))
print("Último partido:", train_2026["date"].max())
print("Primer partido:", train_2026["date"].min())

N entrenamiento 2026: 7313
Último partido: 2026-05-31 00:00:00
Primer partido: 2010-01-02 00:00:00


In [81]:
# Evaluación diagnóstica IN-SAMPLE

elo_cols = [
    "elo_diff_m0",
    "elo_diff_m1",
    "elo_diff_m2",
    "elo_diff_m3",
    "elo_diff_m4",
    "elo_diff_m5"
]

X_2026 = train_2026[elo_cols].to_numpy()
y_2026 = train_2026["outcome"].to_numpy()





mean_2026, cov_2026, inv_sqrt_2026 = fit_whitening(X_2026)

X_white_2026 = transform_whitening(
    X_2026,
    mean_2026,
    inv_sqrt_2026
)





B_sir_2026, _ = fit_sir(
    X_white_2026,
    y_2026,
    d=2
)

sir_2026 = project_sdr(
    X_white_2026,
    B_sir_2026
)





B_save_2026, _ = fit_save(
    X_white_2026,
    y_2026,
    d=2
)

save_2026 = project_sdr(
    X_white_2026,
    B_save_2026
)

print("B SIR 2026:")
print(B_sir_2026)

print("\nB SAVE 2026:")
print(B_save_2026)

B SIR 2026:
[[-0.48780181 -0.41193744]
 [-0.42547716  0.7782482 ]
 [-0.3654767  -0.2746345 ]
 [-0.38084747  0.19888832]
 [-0.38643768 -0.32686665]
 [-0.39123713  0.05305473]]

B SAVE 2026:
[[-0.49762367 -0.79399444]
 [-0.41922164  0.54769235]
 [-0.3441223   0.19163778]
 [-0.36516963  0.14224805]
 [-0.40205571  0.00556968]
 [-0.40398774  0.11231806]]


In [82]:
# Evaluación diagnóstica IN-SAMPLE

train_2026_sdr = train_2026.copy()

train_2026_sdr["sir_2_1"] = sir_2026[:, 0]
train_2026_sdr["sir_2_2"] = sir_2026[:, 1]

train_2026_sdr["save_2_1"] = save_2026[:, 0]
train_2026_sdr["save_2_2"] = save_2026[:, 1]

print(train_2026_sdr.shape)

print(
    train_2026_sdr[
        [
            "sir_2_1",
            "sir_2_2",
            "save_2_1",
            "save_2_2"
        ]
    ].describe()
)

(7313, 36)
           sir_2_1       sir_2_2      save_2_1      save_2_2
count  7313.000000  7.313000e+03  7.313000e+03  7.313000e+03
mean      0.000000  9.808463e-16  7.772928e-18  5.581934e-16
std       1.000000  1.000000e+00  1.000000e+00  1.000000e+00
min      -4.194438 -8.234233e+00 -4.197863e+00 -1.109461e+01
25%      -0.661593 -4.743457e-01 -6.631895e-01 -1.944617e-01
50%      -0.038074  1.486462e-02 -4.458348e-02  3.352055e-02
75%       0.648340  4.927597e-01  6.431052e-01  2.150508e-01
max       4.338063  7.153042e+00  4.327519e+00  9.621772e+00


In [83]:
# Entrenar a M9 y M11

model_m9_home_2026, model_m9_away_2026, train_m9_2026 = fit_poisson_sdr(
    train_2026_sdr,
    ["sir_2_1", "sir_2_2"]
)

model_m11_home_2026, model_m11_away_2026, train_m11_2026 = fit_poisson_sdr(
    train_2026_sdr,
    ["save_2_1", "save_2_2"]
)

In [84]:
# Evaluación diagnóstica IN-SAMPLE

print("========== M9 HOME ==========")
print(model_m9_home_2026.params)

print("\n========== M9 AWAY ==========")
print(model_m9_away_2026.params)

print("\n========== M11 HOME ==========")
print(model_m11_home_2026.params)

print("\n========== M11 AWAY ==========")
print(model_m11_away_2026.params)

========== M9 HOME ==========
const                   0.238020
sir_2_1                -0.464514
sir_2_2                -0.002202
neutral                -0.161577
home_goals_for_6        0.043123
away_goals_against_6    0.072275
dtype: float64

========== M9 AWAY ==========
const                  -0.339649
sir_2_1                -0.454343
sir_2_2                 0.008574
neutral                 0.172402
away_goals_for_6        0.070848
home_goals_against_6    0.113717
dtype: float64

========== M11 HOME ==========
const                   0.232951
save_2_1               -0.463335
save_2_2                0.007141
neutral                -0.161072
home_goals_for_6        0.044915
away_goals_against_6    0.074207
dtype: float64

========== M11 AWAY ==========
const                  -0.346087
save_2_1               -0.453105
save_2_2                0.015021
neutral                 0.172449
away_goals_for_6        0.073099
home_goals_against_6    0.116457
dtype: float64


In [85]:
# PASO 1 — Información de los equipos al 01/06/2026

cutoff_2026 = pd.Timestamp("2026-06-01")


groups_2026 = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Ivory Coast", "Curaçao", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

qualified_2026 = sorted({
    team
    for teams in groups_2026.values()
    for team in teams
})

In [86]:
# PASO 2 — Verificar nombres de equipos

teams_in_data = set(df_model["home_team"].dropna()) | set(df_model["away_team"].dropna())

missing_teams = [
    team for team in qualified_2026
    if team not in teams_in_data
]

print("Equipos del Mundial que NO aparecen en df_model:")
print(missing_teams)

print("\nNúmero de equipos faltantes:", len(missing_teams))

Equipos del Mundial que NO aparecen en df_model:
[]

Número de equipos faltantes: 0


In [87]:
# PASO 3 — Elo mensual hasta el 01/06/2026

cutoff_2026 = pd.Timestamp("2026-06-01")


df_pre_wc = df_model[
    df_model["date"] < cutoff_2026
].copy()


df_pre_wc = df_pre_wc.sort_values("date").reset_index(drop=True)


teams_wc = qualified_2026





elo = {team: 1500.0 for team in teams_wc}


elo_history = []

for _, row in df_pre_wc.iterrows():

    home = row["home_team"]
    away = row["away_team"]



    if home not in elo:
        elo[home] = 1500.0

    if away not in elo:
        elo[away] = 1500.0

    elo_home_pre = elo[home]
    elo_away_pre = elo[away]


    elo_history.append({
        "date": row["date"],
        "home_team": home,
        "away_team": away,
        "home_elo_pre": elo_home_pre,
        "away_elo_pre": elo_away_pre
    })





    home_score = row["home_score"]
    away_score = row["away_score"]

    if home_score > away_score:
        S_home = 1.0
    elif home_score == away_score:
        S_home = 0.5
    else:
        S_home = 0.0

    S_away = 1.0 - S_home






    if row["neutral"] == 1:
        home_advantage = 0
    else:
        home_advantage = 100





    K = get_k_factor(row["tournament"])





    E_home = 1 / (
        1 + 10 ** (
            -(elo_home_pre + home_advantage - elo_away_pre) / 400
        )
    )





    goal_diff = abs(home_score - away_score)

    if goal_diff <= 1:
        goal_multiplier = 1.0
    elif goal_diff == 2:
        goal_multiplier = 1.5
    else:
        goal_multiplier = (11 + goal_diff) / 8





    elo_change = (
        K
        * goal_multiplier
        * (S_home - E_home)
    )

    elo[home] = elo_home_pre + elo_change
    elo[away] = elo_away_pre - elo_change



elo_history_df = pd.DataFrame(elo_history)

print("Partidos procesados:", len(elo_history_df))
print("Último partido:", elo_history_df["date"].max())

Partidos procesados: 48622
Último partido: 2026-05-31 00:00:00


In [88]:
# PASO 4 — Elo mensual de cada selección

elo_history_df["month"] = elo_history_df["date"].dt.to_period("M")



team_elo_records = []

for _, row in elo_history_df.iterrows():

    team_elo_records.append({
        "date": row["date"],
        "month": row["month"],
        "team": row["home_team"],
        "elo": row["home_elo_pre"]
    })

    team_elo_records.append({
        "date": row["date"],
        "month": row["month"],
        "team": row["away_team"],
        "elo": row["away_elo_pre"]
    })

team_elo_df = pd.DataFrame(team_elo_records)






monthly_elo = (
    team_elo_df
    .sort_values(["team", "date"])
    .groupby(["team", "month"], as_index=False)
    .last()
)

monthly_elo = monthly_elo[
    monthly_elo["team"].isin(qualified_2026)
].copy()





months = pd.period_range(
    start="2025-12",
    end="2026-05",
    freq="M"
)





elo_monthly_wide = (
    monthly_elo
    .pivot(
        index="team",
        columns="month",
        values="elo"
    )
)



elo_monthly_wide = elo_monthly_wide.reindex(
    index=qualified_2026,
    columns=months
)






elo_monthly_wide = elo_monthly_wide.ffill(axis=1)

In [89]:
# PASO 5 — Completar Elo mensual correctamente

start_month = pd.Period("2025-12", freq="M")



initial_elo = (
    team_elo_df[
        (team_elo_df["team"].isin(qualified_2026)) &
        (team_elo_df["date"] < pd.Timestamp("2025-12-01"))
    ]
    .sort_values(["team", "date"])
    .groupby("team")
    .tail(1)
    .set_index("team")["elo"]
)


elo_monthly_complete = elo_monthly_wide.copy()




for team in elo_monthly_complete.index:

    if pd.isna(elo_monthly_complete.loc[team, start_month]):

        if team in initial_elo.index:
            elo_monthly_complete.loc[team, start_month] = initial_elo.loc[team]


elo_monthly_complete = elo_monthly_complete.ffill(axis=1)


print(
    "Número total de NaN:",
    elo_monthly_complete.isna().sum().sum()
)

print(
    "Equipos con algún NaN:"
)

print(
    elo_monthly_complete[
        elo_monthly_complete.isna().any(axis=1)
    ].index.tolist()
)

display(elo_monthly_complete)

Número total de NaN: 0
Equipos con algún NaN:
[]


,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05
team,,,,,,
Algeria,1831.990601,1848.954582,1848.954582,1827.695164,1827.695164,1827.695164
Argentina,2142.385305,2142.385305,2142.385305,2144.566074,2144.566074,2144.566074
Australia,1858.316062,1858.316062,1858.316062,1852.829518,1852.829518,1856.341215
Austria,1858.640582,1858.640582,1858.640582,1853.717300,1853.717300,1853.717300
Belgium,1913.844703,1913.844703,1913.844703,1931.637527,1931.637527,1931.637527
Bosnia and Herzegovina,1598.772028,1598.772028,1598.772028,1615.454362,1615.454362,1621.628322
Brazil,2040.884284,2040.884284,2040.884284,2025.208336,2025.208336,2037.424265
Canada,1842.730411,1842.730411,1842.730411,1847.588270,1847.588270,1847.588270
Cape Verde,1646.909229,1646.909229,1646.909229,1640.273974,1640.273974,1639.512756


In [90]:
# PASO 6 — Crear los 72 partidos y diferencias Elo

from itertools import combinations


fixtures_2026 = []

for group, teams in groups_2026.items():

    for team1, team2 in combinations(teams, 2):

        fixtures_2026.append({
            "group": group,
            "team1": team1,
            "team2": team2
        })

fixtures_2026 = pd.DataFrame(fixtures_2026)

print("Número de partidos:", len(fixtures_2026))












month_map = {
    "elo_diff_m0": pd.Period("2026-05", freq="M"),
    "elo_diff_m1": pd.Period("2026-04", freq="M"),
    "elo_diff_m2": pd.Period("2026-03", freq="M"),
    "elo_diff_m3": pd.Period("2026-02", freq="M"),
    "elo_diff_m4": pd.Period("2026-01", freq="M"),
    "elo_diff_m5": pd.Period("2025-12", freq="M"),
}

for col, month in month_map.items():

    fixtures_2026[col] = (
        fixtures_2026["team1"].map(elo_monthly_complete[month])
        -
        fixtures_2026["team2"].map(elo_monthly_complete[month])
    )





print("\nForma:", fixtures_2026.shape)

display(fixtures_2026.head(12))

Número de partidos: 72

Forma: (72, 9)


,group,team1,team2,elo_diff_m0,elo_diff_m1,elo_diff_m2,elo_diff_m3,elo_diff_m4,elo_diff_m5
0,A,Mexico,South Korea,70.766949,60.186551,60.186551,34.184468,27.285776,28.562222
1,A,Mexico,South Africa,260.271608,247.612217,247.612217,229.249304,222.350613,231.949303
2,A,Mexico,Czech Republic,147.566024,144.408560,144.408560,137.029981,130.131289,131.407734
3,A,South Korea,South Africa,189.504659,187.425667,187.425667,195.064836,195.064836,203.387081
4,A,South Korea,Czech Republic,76.799075,84.222009,84.222009,102.845513,102.845513,102.845513
5,A,South Africa,Czech Republic,-112.705585,-103.203658,-103.203658,-92.219324,-92.219324,-100.541569
6,B,Canada,Bosnia and Herzegovina,225.959948,232.133908,232.133908,243.958384,243.958384,243.958384
7,B,Canada,Qatar,283.758273,268.931026,268.931026,264.073167,264.073167,264.073167
8,B,Canada,Switzerland,-60.766934,-59.151093,-59.151093,-81.125825,-81.125825,-81.125825
9,B,Bosnia and Herzegovina,Qatar,57.798325,36.797118,36.797118,20.114783,20.114783,20.114783


In [91]:
# PASO 7: ÚLTIMOS 6 PARTIDOS DE CADA EQUIPO

cutoff = pd.Timestamp("2026-06-01")

df_form = df_model[df_model["date"] < cutoff].copy()
df_form = df_form.sort_values("date")


home_form = df_form[
    ["date", "home_team", "home_score", "away_score"]
].copy()

home_form["team"] = home_form["home_team"]
home_form["goals_for"] = home_form["home_score"]
home_form["goals_against"] = home_form["away_score"]

away_form = df_form[
    ["date", "away_team", "home_score", "away_score"]
].copy()

away_form["team"] = away_form["away_team"]
away_form["goals_for"] = away_form["away_score"]
away_form["goals_against"] = away_form["home_score"]


team_matches = pd.concat(
    [
        home_form[["date", "team", "goals_for", "goals_against"]],
        away_form[["date", "team", "goals_for", "goals_against"]]
    ],
    ignore_index=True
)

team_matches = team_matches.sort_values(["team", "date"])


last6 = (
    team_matches
    .groupby("team", group_keys=False)
    .tail(6)
)


form_6 = (
    last6
    .groupby("team")
    .agg(
        goals_for_6=("goals_for", "mean"),
        goals_against_6=("goals_against", "mean"),
        matches_used=("goals_for", "count")
    )
)


print("Equipos con información:")
print(len(form_6))

print("\nEquipos que no tienen 6 partidos:")
display(form_6[form_6["matches_used"] < 6])

print("\nEjemplo:")
display(form_6.loc[
    ["Mexico", "South Korea", "Brazil", "Argentina", "Spain"]
])

Equipos con información:
310

Equipos que no tienen 6 partidos:


,goals_for_6,goals_against_6,matches_used
team,,,
Ambazonia,0.000000,3.000000,4
Arameans Suryoye,2.000000,0.500000,4
Canary Islands,2.000000,0.666667,3
Cascadia,6.000000,3.000000,1
Central Spain,1.000000,0.000000,1
Chameria,2.500000,2.000000,2
Chechnya,1.000000,13.000000,1
County of Nice,2.333333,2.333333,3
Darfur,0.000000,15.250000,4



Ejemplo:


,goals_for_6,goals_against_6,matches_used
team,,,
Mexico,1.500000,0.166667,6
South Korea,1.666667,0.833333,6
Brazil,2.500000,1.500000,6
Argentina,2.666667,0.333333,6
Spain,2.500000,0.333333,6


In [92]:
# Verificar únicamente las 48 selecciones del Mundial 2026

form_2026 = form_6.loc[
    form_6.index.intersection(qualified_2026)
].copy()

print("Equipos del Mundial con información:", len(form_2026))

print("\nEquipos del Mundial que NO tienen 6 partidos:")
display(
    form_2026[form_2026["matches_used"] < 6]
)

print("\nForma de las 48 selecciones:")
display(
    form_2026.sort_index()
)

Equipos del Mundial con información: 48

Equipos del Mundial que NO tienen 6 partidos:


,goals_for_6,goals_against_6,matches_used
team,,,



Forma de las 48 selecciones:


,goals_for_6,goals_against_6,matches_used
team,,,
Algeria,2.000000,0.500000,6
Argentina,2.666667,0.333333,6
Australia,1.166667,1.333333,6
Austria,3.166667,0.500000,6
Belgium,3.000000,1.000000,6
Bosnia and Herzegovina,1.666667,0.833333,6
Brazil,2.500000,1.500000,6
Canada,0.666667,0.500000,6
Cape Verde,1.666667,1.000000,6


In [93]:
# PASO 7.2: AGREGAR FORMA RECIENTE A LOS FIXTURES

fixtures_2026["home_goals_for_6"] = (
    fixtures_2026["team1"]
    .map(form_2026["goals_for_6"])
)

fixtures_2026["home_goals_against_6"] = (
    fixtures_2026["team1"]
    .map(form_2026["goals_against_6"])
)

fixtures_2026["away_goals_for_6"] = (
    fixtures_2026["team2"]
    .map(form_2026["goals_for_6"])
)

fixtures_2026["away_goals_against_6"] = (
    fixtures_2026["team2"]
    .map(form_2026["goals_against_6"])
)


fixtures_2026["neutral"] = 1

print("Forma del dataframe:", fixtures_2026.shape)

display(
    fixtures_2026.head(12)
)

Forma del dataframe: (72, 14)


,group,team1,team2,elo_diff_m0,elo_diff_m1,elo_diff_m2,elo_diff_m3,elo_diff_m4,elo_diff_m5,home_goals_for_6,home_goals_against_6,away_goals_for_6,away_goals_against_6,neutral
0,A,Mexico,South Korea,70.766949,60.186551,60.186551,34.184468,27.285776,28.562222,1.500000,0.166667,1.666667,0.833333,1
1,A,Mexico,South Africa,260.271608,247.612217,247.612217,229.249304,222.350613,231.949303,1.500000,0.166667,1.000000,1.333333,1
2,A,Mexico,Czech Republic,147.566024,144.408560,144.408560,137.029981,130.131289,131.407734,1.500000,0.166667,2.333333,1.166667,1
3,A,South Korea,South Africa,189.504659,187.425667,187.425667,195.064836,195.064836,203.387081,1.666667,0.833333,1.000000,1.333333,1
4,A,South Korea,Czech Republic,76.799075,84.222009,84.222009,102.845513,102.845513,102.845513,1.666667,0.833333,2.333333,1.166667,1
5,A,South Africa,Czech Republic,-112.705585,-103.203658,-103.203658,-92.219324,-92.219324,-100.541569,1.000000,1.333333,2.333333,1.166667,1
6,B,Canada,Bosnia and Herzegovina,225.959948,232.133908,232.133908,243.958384,243.958384,243.958384,0.666667,0.500000,1.666667,0.833333,1
7,B,Canada,Qatar,283.758273,268.931026,268.931026,264.073167,264.073167,264.073167,0.666667,0.500000,0.666667,1.500000,1
8,B,Canada,Switzerland,-60.766934,-59.151093,-59.151093,-81.125825,-81.125825,-81.125825,0.666667,0.500000,2.000000,1.166667,1
9,B,Bosnia and Herzegovina,Qatar,57.798325,36.797118,36.797118,20.114783,20.114783,20.114783,1.666667,0.833333,0.666667,1.500000,1


In [94]:
# PASO 8: PROYECCIÓN SDR DE LOS 72 PARTIDOS

elo_cols = [
    "elo_diff_m0",
    "elo_diff_m1",
    "elo_diff_m2",
    "elo_diff_m3",
    "elo_diff_m4",
    "elo_diff_m5"
]

X_fixtures = fixtures_2026[elo_cols].to_numpy(dtype=float)


X_white_fixtures = transform_whitening(
    X_fixtures,
    mean_2026,
    inv_sqrt_2026
)


sir_fixtures = project_sdr(
    X_white_fixtures,
    B_sir_2026
)

fixtures_2026["sir_2_1"] = sir_fixtures[:, 0]
fixtures_2026["sir_2_2"] = sir_fixtures[:, 1]


save_fixtures = project_sdr(
    X_white_fixtures,
    B_save_2026
)

fixtures_2026["save_2_1"] = save_fixtures[:, 0]
fixtures_2026["save_2_2"] = save_fixtures[:, 1]

print("Forma:", fixtures_2026.shape)

display(
    fixtures_2026[
        [
            "group",
            "team1",
            "team2",
            "sir_2_1",
            "sir_2_2",
            "save_2_1",
            "save_2_2"
        ]
    ].head(12)
)

Forma: (72, 18)


,group,team1,team2,sir_2_1,sir_2_2,save_2_1,save_2_2
0,A,Mexico,South Korea,-0.144379,-0.588764,-0.127311,-0.612753
1,A,Mexico,South Africa,-0.893068,-0.523683,-0.884338,-0.665603
2,A,Mexico,Czech Republic,-0.451250,-0.048901,-0.437610,-0.105748
3,A,South Korea,South Africa,-0.637394,0.094205,-0.644345,-0.022040
4,A,South Korea,Czech Republic,-0.195576,0.568987,-0.197616,0.537815
5,A,South Africa,Czech Republic,0.553113,0.503905,0.559412,0.590665
6,B,Canada,Bosnia and Herzegovina,-0.768238,0.441814,-0.767136,0.495621
7,B,Canada,Qatar,-1.000918,-0.716992,-1.009881,-0.863916
8,B,Canada,Switzerland,0.370153,-0.119043,0.385434,0.119430
9,B,Bosnia and Herzegovina,Qatar,-0.121386,-1.129682,-0.130062,-1.328727


In [95]:
# Evaluación diagnóstica IN-SAMPLE

print("NaN en variables SDR:")

display(
    fixtures_2026[
        ["sir_2_1", "sir_2_2", "save_2_1", "save_2_2"]
    ].isna().sum()
)

print("\nRangos:")

display(
    fixtures_2026[
        ["sir_2_1", "sir_2_2", "save_2_1", "save_2_2"]
    ].agg(["min", "max", "mean", "std"]).T
)

NaN en variables SDR:


,0
sir_2_1,0
sir_2_2,0
save_2_1,0
save_2_2,0



Rangos:


,min,max,mean,std
sir_2_1,-1.841395,1.643409,-0.246570,0.812137
sir_2_2,-1.129682,0.823617,-0.047360,0.445455
save_2_1,-1.839887,1.640706,-0.241674,0.813007
save_2_2,-1.328727,1.038926,0.024303,0.545531


In [96]:
# PASO 9: MÉXICO VS COREA DEL SUR

match_mexico = fixtures_2026.iloc[0]


p_m9_mexico = predict_match_poisson_sdr(
    model_m9_home_2026,
    model_m9_away_2026,
    match_mexico,
    ["sir_2_1", "sir_2_2"]
)


p_m11_mexico = predict_match_poisson_sdr(
    model_m11_home_2026,
    model_m11_away_2026,
    match_mexico,
    ["save_2_1", "save_2_2"]
)

print("México vs South Korea")
print()
print("M9 - SIR")
print(f"México : {p_m9_mexico[0]:.4f} ({p_m9_mexico[0]*100:.1f}%)")
print(f"Empate : {p_m9_mexico[1]:.4f} ({p_m9_mexico[1]*100:.1f}%)")
print(f"Corea  : {p_m9_mexico[2]:.4f} ({p_m9_mexico[2]*100:.1f}%)")

print()
print("M11 - SAVE")
print(f"México : {p_m11_mexico[0]:.4f} ({p_m11_mexico[0]*100:.1f}%)")
print(f"Empate : {p_m11_mexico[1]:.4f} ({p_m11_mexico[1]*100:.1f}%)")
print(f"Corea  : {p_m11_mexico[2]:.4f} ({p_m11_mexico[2]*100:.1f}%)")

México vs South Korea

M9 - SIR
México : 0.4576 (45.8%)
Empate : 0.2821 (28.2%)
Corea  : 0.2603 (26.0%)

M11 - SAVE
México : 0.4502 (45.0%)
Empate : 0.2838 (28.4%)
Corea  : 0.2661 (26.6%)


In [97]:
# PASO 9.1: SIMETRIZACIÓN MÉXICO vs SOUTH KOREA

row = fixtures_2026.iloc[0].copy()







p_m9_1 = predict_match_poisson_sdr(
    model_m9_home_2026,
    model_m9_away_2026,
    row,
    ["sir_2_1", "sir_2_2"]
)

p_m11_1 = predict_match_poisson_sdr(
    model_m11_home_2026,
    model_m11_away_2026,
    row,
    ["save_2_1", "save_2_2"]
)








row_reverse = row.copy()


row_reverse["home_goals_for_6"] = row["away_goals_for_6"]
row_reverse["home_goals_against_6"] = row["away_goals_against_6"]

row_reverse["away_goals_for_6"] = row["home_goals_for_6"]
row_reverse["away_goals_against_6"] = row["home_goals_against_6"]


for col in elo_cols:
    row_reverse[col] = -row[col]


X_reverse = row_reverse[elo_cols].to_numpy(dtype=float).reshape(1, -1)

X_reverse_white = transform_whitening(
    X_reverse,
    mean_2026,
    inv_sqrt_2026
)

sir_reverse = project_sdr(
    X_reverse_white,
    B_sir_2026
)

save_reverse = project_sdr(
    X_reverse_white,
    B_save_2026
)

row_reverse["sir_2_1"] = sir_reverse[0, 0]
row_reverse["sir_2_2"] = sir_reverse[0, 1]

row_reverse["save_2_1"] = save_reverse[0, 0]
row_reverse["save_2_2"] = save_reverse[0, 1]








p_m9_2 = predict_match_poisson_sdr(
    model_m9_home_2026,
    model_m9_away_2026,
    row_reverse,
    ["sir_2_1", "sir_2_2"]
)

p_m11_2 = predict_match_poisson_sdr(
    model_m11_home_2026,
    model_m11_away_2026,
    row_reverse,
    ["save_2_1", "save_2_2"]
)






p_m9_2_mexico = np.array([
    p_m9_2[2],
    p_m9_2[1],
    p_m9_2[0]
])

p_m11_2_mexico = np.array([
    p_m11_2[2],
    p_m11_2[1],
    p_m11_2[0]
])






p_m9_final = (
    np.array(p_m9_1) + p_m9_2_mexico
) / 2

p_m11_final = (
    np.array(p_m11_1) + p_m11_2_mexico
) / 2


print("========================================")
print("MÉXICO vs SOUTH KOREA")
print("========================================")

print("\nM9 - SIR")
print(
    f"México : {p_m9_final[0]:.4f} "
    f"({p_m9_final[0]*100:.1f}%)"
)
print(
    f"Empate : {p_m9_final[1]:.4f} "
    f"({p_m9_final[1]*100:.1f}%)"
)
print(
    f"Corea  : {p_m9_final[2]:.4f} "
    f"({p_m9_final[2]*100:.1f}%)"
)

print("\nM11 - SAVE")
print(
    f"México : {p_m11_final[0]:.4f} "
    f"({p_m11_final[0]*100:.1f}%)"
)
print(
    f"Empate : {p_m11_final[1]:.4f} "
    f"({p_m11_final[1]*100:.1f}%)"
)
print(
    f"Corea  : {p_m11_final[2]:.4f} "
    f"({p_m11_final[2]*100:.1f}%)"
)

MÉXICO vs SOUTH KOREA

M9 - SIR
México : 0.4351 (43.5%)
Empate : 0.2853 (28.5%)
Corea  : 0.2796 (28.0%)

M11 - SAVE
México : 0.4279 (42.8%)
Empate : 0.2865 (28.7%)
Corea  : 0.2856 (28.6%)


In [98]:
# Evaluación diagnóstica IN-SAMPLE

print("\n--- ORIENTACIÓN 1: México home ---")
print("M9 :", np.round(p_m9_1, 4))
print("M11:", np.round(p_m11_1, 4))

print("\n--- ORIENTACIÓN 2: Corea home ---")
print("M9 :", np.round(p_m9_2, 4))
print("M11:", np.round(p_m11_2, 4))

print("\n--- ORIENTACIÓN 2 convertida a México ---")
print("M9 :", np.round(p_m9_2_mexico, 4))
print("M11:", np.round(p_m11_2_mexico, 4))


--- ORIENTACIÓN 1: México home ---
M9 : [0.4576 0.2821 0.2603]
M11: [0.4502 0.2838 0.2661]

--- ORIENTACIÓN 2: Corea home ---
M9 : [0.2988 0.2886 0.4126]
M11: [0.3051 0.2893 0.4056]

--- ORIENTACIÓN 2 convertida a México ---
M9 : [0.4126 0.2886 0.2988]
M11: [0.4056 0.2893 0.3051]


In [99]:
# PASO 10: AUDITORÍA DEL ELO MENSUAL

df_elo_audit = df_pre_wc.sort_values("date").copy()


ratings = {}


elo_records = []

for _, row in df_elo_audit.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    home_elo = ratings.get(home, 1500.0)
    away_elo = ratings.get(away, 1500.0)


    home_pre = home_elo
    away_pre = away_elo


    K = get_k_factor(row["tournament"])


    home_advantage = 0 if row["neutral"] == 1 else 100


    expected_home = 1 / (
        1 + 10 ** ((away_elo - (home_elo + home_advantage)) / 400)
    )


    if row["home_score"] > row["away_score"]:
        actual_home = 1
    elif row["home_score"] < row["away_score"]:
        actual_home = 0
    else:
        actual_home = 0.5


    goal_diff = abs(
        row["home_score"] - row["away_score"]
    )

    if goal_diff <= 1:
        multiplier = 1
    elif goal_diff == 2:
        multiplier = 1.5
    else:
        multiplier = (11 + goal_diff) / 8

    delta = K * multiplier * (
        actual_home - expected_home
    )

    home_elo = home_elo + delta
    away_elo = away_elo - delta


    elo_records.append({
        "date": row["date"],
        "home_team": home,
        "away_team": away,

        "home_elo_pre": home_pre,
        "away_elo_pre": away_pre,

        "home_elo_post": home_elo,
        "away_elo_post": away_elo,

        "tournament": row["tournament"]
    })


    ratings[home] = home_elo
    ratings[away] = away_elo

elo_audit = pd.DataFrame(elo_records)

print("Registros:", len(elo_audit))
print(
    "Última fecha:",
    elo_audit["date"].max()
)

Registros: 48622
Última fecha: 2026-05-31 00:00:00


In [100]:
# ELO AL CIERRE DE CADA MES

team_post_records = []

for _, row in elo_audit.iterrows():

    team_post_records.append({
        "date": row["date"],
        "team": row["home_team"],
        "elo": row["home_elo_post"]
    })

    team_post_records.append({
        "date": row["date"],
        "team": row["away_team"],
        "elo": row["away_elo_post"]
    })

team_post = pd.DataFrame(team_post_records)

team_post["month"] = (
    team_post["date"]
    .dt.to_period("M")
)

monthly_elo_post = (
    team_post
    .sort_values(["team", "date"])
    .groupby(["team", "month"])
    .tail(1)
)

monthly_elo_post = (
    monthly_elo_post
    .set_index(["team", "month"])["elo"]
    .sort_index()
)


display(
    monthly_elo_post.loc[
        [
            "Mexico",
            "South Korea"
        ]
    ]
)

team         month  
Mexico       1923-12    1513.733079
             1928-05    1498.838343
             1928-06    1475.850975
             1930-07    1327.706165
             1934-03    1399.637657
                           ...     
South Korea  2025-09    1863.255823
             2025-10    1857.460584
             2025-11    1865.103450
             2026-03    1832.976533
             2026-05    1838.806124
Name: elo, Length: 817, dtype: float64

In [101]:
# COMPARACIÓN CORRECTA DEL ELO AL CIERRE DE CADA MES

months = pd.period_range(
    start="2025-12",
    end="2026-05",
    freq="M"
)

teams_check = ["Mexico", "South Korea"]

comparison = []

for team in teams_check:

    team_data = (
        monthly_elo_post
        .loc[team]
        .sort_index()
    )


    team_data = team_data.reindex(months)


    team_data = team_data.ffill()

    for month in months:

        comparison.append({
            "team": team,
            "month": str(month),
            "elo_post": team_data.loc[month]
        })

comparison = pd.DataFrame(comparison)

display(comparison)

,team,month,elo_post
0,Mexico,2025-12,NaN
1,Mexico,2026-01,1896.888710
2,Mexico,2026-02,1900.654740
3,Mexico,2026-03,1900.912572
4,Mexico,2026-04,1900.912572
5,Mexico,2026-05,1912.394408
6,South Korea,2025-12,NaN
7,South Korea,2026-01,NaN
8,South Korea,2026-02,NaN
9,South Korea,2026-03,1832.976533


In [102]:
# ELO INICIAL PARA EL SNAPSHOT POST-MATCH

dec_start = pd.Timestamp("2025-12-01")

initial_elo_post = (
    elo_audit[
        elo_audit["date"] < dec_start
    ]
    .sort_values("date")
)


initial_home = (
    initial_elo_post
    .groupby("home_team")
    .tail(1)
    .set_index("home_team")["home_elo_post"]
)

initial_away = (
    initial_elo_post
    .groupby("away_team")
    .tail(1)
    .set_index("away_team")["away_elo_post"]
)


initial_elo_post = pd.concat(
    [initial_home, initial_away]
)

initial_elo_post = (
    initial_elo_post
    .groupby(level=0)
    .last()
)

display(
    initial_elo_post.loc[
        ["Mexico", "South Korea"]
    ]
)

,0
Mexico,1907.247090
South Korea,1863.255823


In [103]:
# ELO POST-MATCH CORRECTAMENTE ARRASTRADO

months = pd.period_range(
    start="2025-12",
    end="2026-05",
    freq="M"
)

post_comparison = []

for team in ["Mexico", "South Korea"]:


    team_data = (
        monthly_elo_post
        .loc[team]
        .sort_index()
    )

    team_data = team_data.reindex(months)


    if team in initial_elo_post.index:
        team_data.loc[pd.Period("2025-12", freq="M")] = (
            initial_elo_post.loc[team]
        )


    team_data = team_data.ffill()

    for month in months:
        post_comparison.append({
            "team": team,
            "month": str(month),
            "elo_post": team_data.loc[month]
        })

post_comparison = pd.DataFrame(post_comparison)

display(post_comparison)

,team,month,elo_post
0,Mexico,2025-12,1907.247090
1,Mexico,2026-01,1896.888710
2,Mexico,2026-02,1900.654740
3,Mexico,2026-03,1900.912572
4,Mexico,2026-04,1900.912572
5,Mexico,2026-05,1912.394408
6,South Korea,2025-12,1863.255823
7,South Korea,2026-01,1863.255823
8,South Korea,2026-02,1863.255823
9,South Korea,2026-03,1832.976533


In [104]:
# DIFERENCIA MÉXICO - COREA

post_wide = (
    post_comparison
    .pivot(
        index="month",
        columns="team",
        values="elo_post"
    )
    .reset_index()
)

post_wide["Mexico_minus_Korea"] = (
    post_wide["Mexico"]
    - post_wide["South Korea"]
)

display(post_wide)

team,month,Mexico,South Korea,Mexico_minus_Korea
0,2025-12,1907.247090,1863.255823,43.991267
1,2026-01,1896.888710,1863.255823,33.632887
2,2026-02,1900.654740,1863.255823,37.398918
3,2026-03,1900.912572,1832.976533,67.936039
4,2026-04,1900.912572,1832.976533,67.936039
5,2026-05,1912.394408,1838.806124,73.588284


In [105]:
# COMPARAR SNAPSHOT ACTUAL VS SNAPSHOT POST-MATCH

audit_final = pd.DataFrame({
    "month": post_wide["month"],

    "Mexico_current": [
        elo_monthly_complete.loc[
            "Mexico",
            pd.Period(m, freq="M")
        ]
        for m in post_wide["month"]
    ],

    "Korea_current": [
        elo_monthly_complete.loc[
            "South Korea",
            pd.Period(m, freq="M")
        ]
        for m in post_wide["month"]
    ],

    "Mexico_post": post_wide["Mexico"].values,

    "Korea_post": post_wide["South Korea"].values,

    "diff_current": [
        elo_monthly_complete.loc[
            "Mexico",
            pd.Period(m, freq="M")
        ]
        -
        elo_monthly_complete.loc[
            "South Korea",
            pd.Period(m, freq="M")
        ]
        for m in post_wide["month"]
    ],

    "diff_post": post_wide["Mexico_minus_Korea"].values
})

audit_final["change_in_diff"] = (
    audit_final["diff_post"]
    - audit_final["diff_current"]
)

display(audit_final)

,month,Mexico_current,Korea_current,Mexico_post,Korea_post,diff_current,diff_post,change_in_diff
0,2025-12,1891.266488,1862.704267,1907.247090,1863.255823,28.562222,43.991267,15.429045
1,2026-01,1889.990043,1862.704267,1896.888710,1863.255823,27.285776,33.632887,6.347111
2,2026-02,1896.888735,1862.704267,1900.654740,1863.255823,34.184468,37.398918,3.214450
3,2026-03,1900.004389,1839.817839,1900.912572,1832.976533,60.186551,67.936039,7.749488
4,2026-04,1900.004389,1839.817839,1900.912572,1832.976533,60.186551,67.936039,7.749488
5,2026-05,1903.750475,1832.983526,1912.394408,1838.806124,70.766949,73.588284,2.821335


In [106]:
# PASO 10: RECONSTRUIR LAGS MENSUALES CON ELO POST-MATCH

df_model_corrected = df_model.copy()


df_model_corrected["month"] = (
    df_model_corrected["date"].dt.to_period("M")
)





monthly_elo_post_dict = {}

for _, row in df_model_corrected.iterrows():

    month = row["month"]

    if month not in monthly_elo_post_dict:
        monthly_elo_post_dict[month] = {}

    monthly_elo_post_dict[month][
        row["home_team"]
    ] = row["home_elo_post"]

    monthly_elo_post_dict[month][
        row["away_team"]
    ] = row["away_elo_post"]






def get_team_elo_post_for_month(team, month):

    available_months = [
        m
        for m in monthly_elo_post_dict.keys()
        if m <= month and team in monthly_elo_post_dict[m]
    ]

    if not available_months:
        return np.nan

    last_month = max(available_months)

    return monthly_elo_post_dict[last_month][team]

In [107]:
# CONSTRUIR m0 ... m5

def build_lagged_elo_features_corrected(row, n_lags=6):

    current_month = row["month"]

    values = []





    values.append(
        row["home_elo"] - row["away_elo"]
    )





    for lag in range(1, n_lags):

        target_month = current_month - lag

        home_elo = get_team_elo_post_for_month(
            row["home_team"],
            target_month
        )

        away_elo = get_team_elo_post_for_month(
            row["away_team"],
            target_month
        )

        if pd.isna(home_elo) or pd.isna(away_elo):
            values.append(np.nan)
        else:
            values.append(
                home_elo - away_elo
            )

    return values

In [108]:
# LAGS MENSUALES DE ELO — VERSIÓN RÁPIDA

elo_lag_cols = [
    "elo_diff_m0",
    "elo_diff_m1",
    "elo_diff_m2",
    "elo_diff_m3",
    "elo_diff_m4",
    "elo_diff_m5"
]


df_model_corrected = df_model_corrected.sort_values(
    "date"
).reset_index(drop=True)

df_model_corrected["month"] = (
    df_model_corrected["date"].dt.to_period("M")
)





df_model_corrected["elo_diff_m0"] = (
    df_model_corrected["home_elo"]
    - df_model_corrected["away_elo"]
)





home_post = df_model_corrected[
    ["month", "home_team", "home_elo_post"]
].rename(
    columns={
        "home_team": "team",
        "home_elo_post": "elo_post"
    }
)

away_post = df_model_corrected[
    ["month", "away_team", "away_elo_post"]
].rename(
    columns={
        "away_team": "team",
        "away_elo_post": "elo_post"
    }
)

team_month_elo = pd.concat(
    [home_post, away_post],
    ignore_index=True
)



team_month_elo = (
    team_month_elo
    .sort_values(["team", "month"])
    .drop_duplicates(
        ["team", "month"],
        keep="last"
    )
)





all_months = pd.period_range(
    df_model_corrected["month"].min(),
    df_model_corrected["month"].max(),
    freq="M"
)

teams = pd.unique(
    pd.concat([
        df_model_corrected["home_team"],
        df_model_corrected["away_team"]
    ])
)

full_index = pd.MultiIndex.from_product(
    [teams, all_months],
    names=["team", "month"]
)

team_month_elo = (
    team_month_elo
    .set_index(["team", "month"])
    .reindex(full_index)
    .sort_index()
)






team_month_elo["elo_post"] = (
    team_month_elo
    .groupby(level="team")["elo_post"]
    .ffill()
)

team_month_elo = team_month_elo.reset_index()





def add_month_lag(df, lag):


    target_month = (
        df["month"] - lag
    )


    home_lookup = pd.DataFrame({
        "team": df["home_team"].values,
        "month": target_month.values
    })

    home_lookup["row_id"] = np.arange(len(df))

    home_lookup = home_lookup.merge(
        team_month_elo,
        on=["team", "month"],
        how="left"
    )


    away_lookup = pd.DataFrame({
        "team": df["away_team"].values,
        "month": target_month.values
    })

    away_lookup["row_id"] = np.arange(len(df))

    away_lookup = away_lookup.merge(
        team_month_elo,
        on=["team", "month"],
        how="left"
    )

    return (
        home_lookup.sort_values("row_id")["elo_post"].to_numpy()
        -
        away_lookup.sort_values("row_id")["elo_post"].to_numpy()
    )






for lag in range(1, 6):

    col = f"elo_diff_m{lag}"

    df_model_corrected[col] = add_month_lag(
        df_model_corrected,
        lag
    )

print("Listo.")

Listo.


In [109]:
# ELO ACTUAL DE LOS 48 EQUIPOS AL CORTE 2026-06-01

qualified_2026 = [
    team
    for group in groups_2026.values()
    for team in group
]

latest_elo_2026 = (
    team_month_elo[
        (team_month_elo["team"].isin(qualified_2026)) &
        (team_month_elo["month"] <= pd.Period("2026-05", freq="M"))
    ]
    .sort_values(["team", "month"])
    .groupby("team")
    .tail(1)
    [["team", "month", "elo_post"]]
    .sort_values("team")
)

print(latest_elo_2026.to_string(index=False))
print("\nEquipos:", len(latest_elo_2026))

                  team   month    elo_post
               Algeria 2026-05 1833.309047
             Argentina 2026-05 2148.241376
             Australia 2026-05 1850.016779
               Austria 2026-05 1863.952962
               Belgium 2026-05 1934.074177
Bosnia and Herzegovina 2026-05 1621.909232
                Brazil 2026-05 2045.052492
                Canada 2026-05 1844.960180
            Cape Verde 2026-05 1667.023989
              Colombia 2026-05 2021.728467
               Croatia 2026-05 1951.029884
               Curaçao 2026-05 1551.979706
        Czech Republic 2026-05 1765.251638
              DR Congo 2026-05 1708.741997
               Ecuador 2026-05 1958.528914
                 Egypt 2026-05 1786.935003
               England 2026-05 2018.193209
                France 2026-05 2102.115108
               Germany 2026-05 1961.517251
                 Ghana 2026-05 1608.289209
                 Haiti 2026-05 1649.212568
                  Iran 2026-05 1860.986018
           

In [110]:
# Construcción

fixtures_2026 = []

for group, teams in groups_2026.items():
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            fixtures_2026.append({
                "group": group,
                "home_team": teams[i],
                "away_team": teams[j],
                "neutral": 1
            })

fixtures_2026 = pd.DataFrame(fixtures_2026)

print("Partidos:", len(fixtures_2026))
display(fixtures_2026)

Partidos: 72


,group,home_team,away_team,neutral
0,A,Mexico,South Korea,1
1,A,Mexico,South Africa,1
2,A,Mexico,Czech Republic,1
3,A,South Korea,South Africa,1
4,A,South Korea,Czech Republic,1
...,...,...,...,...
67,L,England,Ghana,1
68,L,England,Panama,1
69,L,Croatia,Ghana,1
70,L,Croatia,Panama,1


In [111]:
# ELO M0-M5 PARA LOS PARTIDOS DE 2026

forecast_months = {
    0: pd.Period("2026-05", freq="M"),
    1: pd.Period("2026-04", freq="M"),
    2: pd.Period("2026-03", freq="M"),
    3: pd.Period("2026-02", freq="M"),
    4: pd.Period("2026-01", freq="M"),
    5: pd.Period("2025-12", freq="M"),
}

elo_lookup = (
    team_month_elo
    .set_index(["team", "month"])["elo_post"]
)

for lag in range(6):

    month = forecast_months[lag]

    home_keys = pd.MultiIndex.from_arrays([
        fixtures_2026["home_team"],
        [month] * len(fixtures_2026)
    ])

    away_keys = pd.MultiIndex.from_arrays([
        fixtures_2026["away_team"],
        [month] * len(fixtures_2026)
    ])

    home_elo = elo_lookup.reindex(home_keys).to_numpy()
    away_elo = elo_lookup.reindex(away_keys).to_numpy()

    fixtures_2026[f"elo_diff_m{lag}"] = (
        home_elo - away_elo
    )

print(fixtures_2026[[
    "group",
    "home_team",
    "away_team",
    "elo_diff_m0",
    "elo_diff_m1",
    "elo_diff_m2",
    "elo_diff_m3",
    "elo_diff_m4",
    "elo_diff_m5"
]].head(12))

   group               home_team               away_team  elo_diff_m0  \
0      A                  Mexico             South Korea    74.003755   
1      A                  Mexico            South Africa   276.016497   
2      A                  Mexico          Czech Republic   149.963824   
3      A             South Korea            South Africa   202.012742   
4      A             South Korea          Czech Republic    75.960068   
5      A            South Africa          Czech Republic  -126.052673   
6      B                  Canada  Bosnia and Herzegovina   223.050947   
7      B                  Canada                   Qatar   282.897506   
8      B                  Canada             Switzerland   -72.463142   
9      B  Bosnia and Herzegovina                   Qatar    59.846559   
10     B  Bosnia and Herzegovina             Switzerland  -295.514090   
11     B                   Qatar             Switzerland  -355.360648   

    elo_diff_m1  elo_diff_m2  elo_diff_m3  elo_dif

In [112]:
# M9 Y M11 — ENTRENAMIENTO 2026

score_cols_M9 = ["sir_2_1", "sir_2_2"]
score_cols_M11 = ["save_2_1", "save_2_2"]

model_M9_home, model_M9_away, data_M9 = fit_poisson_sdr(
    train_2026,
    score_cols_M9
)

model_M11_home, model_M11_away, data_M11 = fit_poisson_sdr(
    train_2026, score_cols_M11)

In [113]:
# ENTRENAMIENTO 2026 + WHITENING

train_2026 = df_model_corrected[
    (df_model_corrected["date"] >= pd.Timestamp("2010-01-01"))
    & (df_model_corrected["date"] < pd.Timestamp("2026-06-01"))
    & (
        df_model_corrected["home_team"].isin(qualified_2026)
        | df_model_corrected["away_team"].isin(qualified_2026)
    )
].copy()

train_2026 = train_2026.dropna(
    subset=elo_lag_cols
).copy()

print("N entrenamiento 2026:", len(train_2026))

X_train_2026 = train_2026[elo_lag_cols].values
y_train_2026 = np.select(
    [
        train_2026["home_score"] > train_2026["away_score"],
        train_2026["home_score"] < train_2026["away_score"]
    ],
    ["H", "A"],
    default="D"
)

mean_2026, cov_2026, inv_sqrt_2026 = fit_whitening(
    X_train_2026
)

X_white_2026 = transform_whitening(
    X_train_2026,
    mean_2026,
    inv_sqrt_2026
)

N entrenamiento 2026: 7312


In [114]:
# SIR d=2

B_sir_2026, _ = fit_sir(
    X_white_2026,
    y_train_2026,
    d=2
)


B_save_2026, _ = fit_save(
    X_white_2026,
    y_train_2026,
    d=2
)

print("B SIR:")
print(B_sir_2026)

print("\nB SAVE:")
print(B_save_2026)

B SIR:
[[-0.48968944 -0.30707582]
 [-0.42571072  0.74710109]
 [-0.36137223 -0.41001307]
 [-0.37137567  0.20798939]
 [-0.3789153   0.1206457 ]
 [-0.40851934 -0.34873874]]

B SAVE:
[[-0.49084236 -0.76707872]
 [-0.42228229  0.60064914]
 [-0.35910523  0.14081049]
 [-0.36022733  0.16857234]
 [-0.39325225 -0.01928797]
 [-0.40912568  0.04684653]]


In [115]:
# PROYECCIÓN DE LOS 72 PARTIDOS

X_fixtures_2026 = fixtures_2026[elo_lag_cols].values

X_fixtures_white_2026 = transform_whitening(
    X_fixtures_2026,
    mean_2026,
    inv_sqrt_2026
)

sir_scores_2026 = (
    X_fixtures_white_2026 @ B_sir_2026
)

save_scores_2026 = (
    X_fixtures_white_2026 @ B_save_2026
)

fixtures_2026["sir_2_1"] = sir_scores_2026[:, 0]
fixtures_2026["sir_2_2"] = sir_scores_2026[:, 1]

fixtures_2026["save_2_1"] = save_scores_2026[:, 0]
fixtures_2026["save_2_2"] = save_scores_2026[:, 1]

print(
    fixtures_2026[
        [
            "group",
            "home_team",
            "away_team",
            "sir_2_1",
            "sir_2_2",
            "save_2_1",
            "save_2_2"
        ]
    ].head(12)
)

   group               home_team               away_team   sir_2_1   sir_2_2  \
0      A                  Mexico             South Korea -0.118463 -0.309037   
1      A                  Mexico            South Africa -0.904498  0.037070   
2      A                  Mexico          Czech Republic -0.436132  0.145002   
3      A             South Korea            South Africa -0.675134  0.358137   
4      A             South Korea          Czech Republic -0.206767  0.466069   
5      A            South Africa          Czech Republic  0.579268  0.119962   
6      B                  Canada  Bosnia and Herzegovina -0.767900  0.447241   
7      B                  Canada                   Qatar -0.999850  0.098504   
8      B                  Canada             Switzerland  0.394518  0.245557   
9      B  Bosnia and Herzegovina                   Qatar -0.121048 -0.336707   
10     B  Bosnia and Herzegovina             Switzerland  1.273319 -0.189654   
11     B                   Qatar        

In [116]:
# POISSON M9 / M11

model_M9_home, model_M9_away, _ = fit_poisson_sdr(
    train_2026,
    ["sir_2_1", "sir_2_2"]
)

model_M11_home, model_M11_away, _ = fit_poisson_sdr(
    train_2026,
    ["save_2_1", "save_2_2"]
)

print("M9 y M11 listos.")

M9 y M11 listos.


In [117]:
# FORMA: ÚLTIMOS 6 PARTIDOS ANTES DEL 1-JUN-2026

cutoff_2026 = pd.Timestamp("2026-06-01")

df_before_cutoff = df_model_corrected[
    df_model_corrected["date"] < cutoff_2026
].copy()

teams_2026 = qualified_2026

form_rows = []

for team in teams_2026:

    home = df_before_cutoff[
        df_before_cutoff["home_team"] == team
    ].copy()

    away = df_before_cutoff[
        df_before_cutoff["away_team"] == team
    ].copy()

    home_form = pd.DataFrame({
        "date": home["date"],
        "goals_for": home["home_score"],
        "goals_against": home["away_score"]
    })

    away_form = pd.DataFrame({
        "date": away["date"],
        "goals_for": away["away_score"],
        "goals_against": away["home_score"]
    })

    team_form = pd.concat(
        [home_form, away_form],
        ignore_index=True
    ).sort_values("date")

    last6 = team_form.tail(6)

    form_rows.append({
        "team": team,
        "goals_for_6": last6["goals_for"].mean(),
        "goals_against_6": last6["goals_against"].mean()
    })

form_2026 = pd.DataFrame(form_rows)

print(form_2026.to_string(index=False))

                  team  goals_for_6  goals_against_6
                Mexico     1.500000         0.166667
           South Korea     1.666667         0.833333
          South Africa     1.000000         1.333333
        Czech Republic     2.333333         1.166667
                Canada     0.666667         0.500000
Bosnia and Herzegovina     1.666667         0.833333
                 Qatar     0.666667         1.500000
           Switzerland     2.000000         1.166667
                Brazil     2.500000         1.500000
               Morocco     2.166667         0.333333
                 Haiti     1.166667         0.833333
              Scotland     2.000000         1.500000
         United States     2.333333         2.000000
              Paraguay     1.166667         1.500000
             Australia     1.166667         1.333333
                Turkey     2.666667         0.666667
               Germany     3.166667         0.666667
           Ivory Coast     2.333333         1.

In [118]:
# AÑADIR FORMA A FIXTURES

home_form = form_2026.rename(
    columns={
        "team": "home_team",
        "goals_for_6": "home_goals_for_6",
        "goals_against_6": "home_goals_against_6"
    }
)

away_form = form_2026.rename(
    columns={
        "team": "away_team",
        "goals_for_6": "away_goals_for_6",
        "goals_against_6": "away_goals_against_6"
    }
)

fixtures_2026 = fixtures_2026.merge(
    home_form[
        [
            "home_team",
            "home_goals_for_6",
            "home_goals_against_6"
        ]
    ],
    on="home_team",
    how="left"
)

fixtures_2026 = fixtures_2026.merge(
    away_form[
        [
            "away_team",
            "away_goals_for_6",
            "away_goals_against_6"
        ]
    ],
    on="away_team",
    how="left"
)

print(
    fixtures_2026[
        [
            "group",
            "home_team",
            "away_team",
            "home_goals_for_6",
            "home_goals_against_6",
            "away_goals_for_6",
            "away_goals_against_6"
        ]
    ].head(12)
)

   group               home_team               away_team  home_goals_for_6  \
0      A                  Mexico             South Korea          1.500000   
1      A                  Mexico            South Africa          1.500000   
2      A                  Mexico          Czech Republic          1.500000   
3      A             South Korea            South Africa          1.666667   
4      A             South Korea          Czech Republic          1.666667   
5      A            South Africa          Czech Republic          1.000000   
6      B                  Canada  Bosnia and Herzegovina          0.666667   
7      B                  Canada                   Qatar          0.666667   
8      B                  Canada             Switzerland          0.666667   
9      B  Bosnia and Herzegovina                   Qatar          1.666667   
10     B  Bosnia and Herzegovina             Switzerland          1.666667   
11     B                   Qatar             Switzerland        

In [119]:
# PROBABILIDADES M9 Y M11

def predict_fixture_model(
    model_home,
    model_away,
    row,
    score_cols
):
    return predict_match_poisson_sdr(
        model_home,
        model_away,
        row,
        score_cols,
        gmax=8
    )


predictions_2026 = []

for _, row in fixtures_2026.iterrows():

    p9 = predict_fixture_model(
        model_M9_home,
        model_M9_away,
        row,
        ["sir_2_1", "sir_2_2"]
    )

    p11 = predict_fixture_model(
        model_M11_home,
        model_M11_away,
        row,
        ["save_2_1", "save_2_2"]
    )

    predictions_2026.append({
        "group": row["group"],
        "home_team": row["home_team"],
        "away_team": row["away_team"],

        "M9_H": p9[0],
        "M9_D": p9[1],
        "M9_A": p9[2],

        "M11_H": p11[0],
        "M11_D": p11[1],
        "M11_A": p11[2],
    })

predictions_2026 = pd.DataFrame(predictions_2026)

print(predictions_2026.head(12).to_string(index=False))

group              home_team              away_team     M9_H     M9_D     M9_A    M11_H    M11_D    M11_A
    A                 Mexico            South Korea 0.427316 0.287868 0.284816 0.425118 0.287794 0.287089
    A                 Mexico           South Africa 0.626045 0.230975 0.142980 0.619945 0.232973 0.147082
    A                 Mexico         Czech Republic 0.495052 0.269452 0.235496 0.494162 0.269361 0.236477
    A            South Korea           South Africa 0.560163 0.250852 0.188986 0.556916 0.251502 0.191582
    A            South Korea         Czech Republic 0.425040 0.276008 0.298952 0.427473 0.275648 0.296880
    A           South Africa         Czech Republic 0.240140 0.261821 0.498038 0.245944 0.263912 0.490144
    B                 Canada Bosnia and Herzegovina 0.549519 0.258290 0.192191 0.552768 0.257527 0.189705
    B                 Canada                  Qatar 0.635356 0.227370 0.137274 0.634029 0.227916 0.138055
    B                 Canada            Switze

In [120]:
# PREDICCIÓN SIMETRIZADA — M9 Y M11

def predict_symmetrized(
    model_home,
    model_away,
    row,
    score_cols
):





    p_original = predict_match_poisson_sdr(
        model_home,
        model_away,
        row,
        score_cols,
        gmax=8
    )









    reverse_row = row.copy()


    for col in score_cols:
        reverse_row[col] = -row[col]


    reverse_row["home_goals_for_6"] = row["away_goals_for_6"]
    reverse_row["home_goals_against_6"] = row["away_goals_against_6"]

    reverse_row["away_goals_for_6"] = row["home_goals_for_6"]
    reverse_row["away_goals_against_6"] = row["home_goals_against_6"]


    p_reverse = predict_match_poisson_sdr(
        model_home,
        model_away,
        reverse_row,
        score_cols,
        gmax=8
    )





    p_home = 0.5 * (
        p_original[0] +
        p_reverse[2]
    )

    p_draw = 0.5 * (
        p_original[1] +
        p_reverse[1]
    )

    p_away = 0.5 * (
        p_original[2] +
        p_reverse[0]
    )


    total = p_home + p_draw + p_away

    return (
        p_home / total,
        p_draw / total,
        p_away / total
    )

In [121]:
# PROBABILIDADES FINALES 2026

predictions_2026 = []

for _, row in fixtures_2026.iterrows():

    p9 = predict_symmetrized(
        model_M9_home,
        model_M9_away,
        row,
        ["sir_2_1", "sir_2_2"]
    )

    p11 = predict_symmetrized(
        model_M11_home,
        model_M11_away,
        row,
        ["save_2_1", "save_2_2"]
    )

    predictions_2026.append({
        "group": row["group"],
        "home_team": row["home_team"],
        "away_team": row["away_team"],

        "M9_H": p9[0],
        "M9_D": p9[1],
        "M9_A": p9[2],

        "M11_H": p11[0],
        "M11_D": p11[1],
        "M11_A": p11[2],
    })

predictions_2026 = pd.DataFrame(predictions_2026)

print(
    predictions_2026.head(12).to_string(index=False)
)

group              home_team              away_team     M9_H     M9_D     M9_A    M11_H    M11_D    M11_A
    A                 Mexico            South Korea 0.395792 0.289419 0.314789 0.392911 0.289575 0.317514
    A                 Mexico           South Africa 0.595478 0.241161 0.163361 0.587685 0.244017 0.168298
    A                 Mexico         Czech Republic 0.466484 0.274618 0.258897 0.464862 0.274754 0.260384
    A            South Korea           South Africa 0.532378 0.258011 0.209611 0.528023 0.259124 0.212854
    A            South Korea         Czech Republic 0.401268 0.278372 0.320360 0.403164 0.277934 0.318902
    A           South Africa         Czech Republic 0.222285 0.257068 0.520648 0.227356 0.258519 0.514125
    B                 Canada Bosnia and Herzegovina 0.512993 0.269377 0.217630 0.516034 0.268539 0.215427
    B                 Canada                  Qatar 0.602049 0.239624 0.158327 0.599951 0.240459 0.159590
    B                 Canada            Switze

In [122]:
# Evaluación diagnóstica IN-SAMPLE

predictions_2026[
    (predictions_2026["home_team"] == "Mexico") &
    (predictions_2026["away_team"] == "South Korea")
]

,group,home_team,away_team,M9_H,M9_D,M9_A,M11_H,M11_D,M11_A
0,A,Mexico,South Korea,0.395792,0.289419,0.314789,0.392911,0.289575,0.317514


In [123]:
# Evaluación diagnóstica IN-SAMPLE

print(
    predictions_2026[
        ["M9_H", "M9_D", "M9_A",
         "M11_H", "M11_D", "M11_A"]
    ].isna().sum()
)

print("\nM9 suma:")
print(
    (
        predictions_2026["M9_H"]
        + predictions_2026["M9_D"]
        + predictions_2026["M9_A"]
    ).describe()
)

print("\nM11 suma:")
print(
    (
        predictions_2026["M11_H"]
        + predictions_2026["M11_D"]
        + predictions_2026["M11_A"]
    ).describe()
)

M9_H     0
M9_D     0
M9_A     0
M11_H    0
M11_D    0
M11_A    0
dtype: int64

M9 suma:
count    7.200000e+01
mean     1.000000e+00
std      6.180056e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

M11 suma:
count    7.200000e+01
mean     1.000000e+00
std      5.270369e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


In [124]:
# PREPARAR PROBABILIDADES PARA LA SIMULACIÓN

match_probs = {}

for _, row in predictions_2026.iterrows():

    key = (
        row["home_team"],
        row["away_team"]
    )

    match_probs[key] = (
        row["M9_H"],
        row["M9_D"],
        row["M9_A"]
    )

print("Partidos preparados:", len(match_probs))

Partidos preparados: 72


In [125]:
# SIMULACIÓN DE UNA FASE DE GRUPOS

rng = np.random.default_rng(42)


def simulate_match(p_home, p_draw, p_away, rng):
    """
    Simula el resultado de un partido:
    H = victoria local
    D = empate
    A = victoria visitante
    """
    return rng.choice(
        ["H", "D", "A"],
        p=[p_home, p_draw, p_away]
    )


def simulate_group_stage(groups, match_probs, rng):
    """
    Simula los 72 partidos de grupos.

    Devuelve:
        results: resultados de los 72 partidos
        standings: tabla de cada grupo
    """

    results = []


    standings = {}

    for group, teams in groups.items():
        standings[group] = {}

        for team in teams:
            standings[group][team] = {
                "points": 0,
                "gf": 0,
                "ga": 0,
                "gd": 0,
                "wins": 0,
                "draws": 0,
                "losses": 0
            }


    for _, match in predictions_2026.iterrows():

        group = match["group"]
        home = match["home_team"]
        away = match["away_team"]

        p_home = match["M9_H"]
        p_draw = match["M9_D"]
        p_away = match["M9_A"]

        result = simulate_match(
            p_home,
            p_draw,
            p_away,
            rng
        )


        results.append({
            "group": group,
            "home_team": home,
            "away_team": away,
            "result": result
        })







        if result == "H":
            hg, ag = 1, 0
        elif result == "D":
            hg, ag = 1, 1
        else:
            hg, ag = 0, 1

        h = standings[group][home]
        a = standings[group][away]

        h["gf"] += hg
        h["ga"] += ag
        h["gd"] = h["gf"] - h["ga"]

        a["gf"] += ag
        a["ga"] += hg
        a["gd"] = a["gf"] - a["ga"]

        if result == "H":
            h["points"] += 3
            h["wins"] += 1
            a["losses"] += 1

        elif result == "D":
            h["points"] += 1
            a["points"] += 1
            h["draws"] += 1
            a["draws"] += 1

        else:
            a["points"] += 3
            a["wins"] += 1
            h["losses"] += 1


    standings_rows = []

    for group, teams in standings.items():

        for team, stats in teams.items():

            standings_rows.append({
                "group": group,
                "team": team,
                **stats
            })

    standings_df = pd.DataFrame(standings_rows)

    return pd.DataFrame(results), standings_df


group_results, standings = simulate_group_stage(
    groups_2026,
    match_probs,
    rng
)

print("Partidos simulados:", len(group_results))
print("Equipos:", len(standings))

display(
    standings.sort_values(
        ["group", "points", "gd", "gf"],
        ascending=[True, False, False, False]
    )
)

Partidos simulados: 72
Equipos: 48


,group,team,points,gf,ga,gd,wins,draws,losses
1,A,South Korea,7,3,1,2,2,1,0
3,A,Czech Republic,6,2,1,1,2,0,1
0,A,Mexico,3,1,2,-1,1,0,2
2,A,South Africa,1,1,3,-2,0,1,2
7,B,Switzerland,6,2,1,1,2,0,1
4,B,Canada,5,3,2,1,1,2,0
5,B,Bosnia and Herzegovina,2,2,3,-1,0,2,1
6,B,Qatar,2,2,3,-1,0,2,1
8,C,Brazil,5,3,2,1,1,2,0
9,C,Morocco,5,3,2,1,1,2,0


In [126]:
# Evaluación diagnóstica IN-SAMPLE

from scipy.stats import poisson

def get_score_matrix(
    model_home,
    model_away,
    row,
    score_cols,
    gmax=8
):



    home_cols = score_cols + [
        "neutral",
        "home_goals_for_6",
        "away_goals_against_6"
    ]

    X_home = pd.DataFrame(
        [[row[col] for col in home_cols]],
        columns=home_cols
    )

    X_home = sm.add_constant(
        X_home,
        has_constant="add"
    )




    away_cols = score_cols + [
        "neutral",
        "away_goals_for_6",
        "home_goals_against_6"
    ]

    X_away = pd.DataFrame(
        [[row[col] for col in away_cols]],
        columns=away_cols
    )

    for col in score_cols:
        X_away[col] = -X_away[col]

    X_away = sm.add_constant(
        X_away,
        has_constant="add"
    )

    X_home = np.asarray(
        X_home,
        dtype=np.float64
    )

    X_away = np.asarray(
        X_away,
        dtype=np.float64
    )

    lambda_home = float(
        model_home.predict(X_home)[0]
    )

    lambda_away = float(
        model_away.predict(X_away)[0]
    )

    goals = np.arange(gmax + 1)

    p_home_goals = poisson.pmf(
        goals,
        lambda_home
    )

    p_away_goals = poisson.pmf(
        goals,
        lambda_away
    )

    score_matrix = np.outer(
        p_home_goals,
        p_away_goals
    )


    score_matrix = (
        score_matrix /
        score_matrix.sum()
    )

    return (
        lambda_home,
        lambda_away,
        score_matrix
    )

In [127]:
# Evaluación diagnóstica IN-SAMPLE

mexico_korea = fixtures_2026[
    (fixtures_2026["home_team"] == "Mexico") &
    (fixtures_2026["away_team"] == "South Korea")
].iloc[0]

lambda_h, lambda_a, score_matrix = get_score_matrix(
    model_M9_home,
    model_M9_away,
    mexico_korea,
    ["sir_2_1", "sir_2_2"]
)

print("México lambda:", lambda_h)
print("Corea lambda:", lambda_a)

print("\nSuma matriz:", score_matrix.sum())

México lambda: 1.2392382924482654
Corea lambda: 0.9553797866889631

Suma matriz: 1.0


In [128]:
# M3 — ELO-ONLY POISSON FALLBACK 2026

model_M3_home, model_M3_away, data_M3 = fit_poisson_sdr(
    train_2026,
    ["elo_diff_m0"]
)

print("M3 ajustado.")

M3 ajustado.


In [129]:
# M3 SIMETRIZADO

def predict_m3_symmetrized(model_home, model_away, row):


    p_original = predict_match_poisson_sdr(
        model_home,
        model_away,
        row,
        ["elo_diff_m0"],
        gmax=8
    )


    reverse_row = row.copy()

    reverse_row["elo_diff_m0"] = -row["elo_diff_m0"]

    reverse_row["home_goals_for_6"] = row["away_goals_for_6"]
    reverse_row["home_goals_against_6"] = row["away_goals_against_6"]

    reverse_row["away_goals_for_6"] = row["home_goals_for_6"]
    reverse_row["away_goals_against_6"] = row["home_goals_against_6"]

    p_reverse = predict_match_poisson_sdr(
        model_home,
        model_away,
        reverse_row,
        ["elo_diff_m0"],
        gmax=8
    )

    p_home = 0.5 * (
        p_original[0] +
        p_reverse[2]
    )

    p_draw = 0.5 * (
        p_original[1] +
        p_reverse[1]
    )

    p_away = 0.5 * (
        p_original[2] +
        p_reverse[0]
    )

    total = p_home + p_draw + p_away

    return (
        p_home / total,
        p_draw / total,
        p_away / total
    )

In [130]:
# Evaluación diagnóstica IN-SAMPLE

curacao_matches = predictions_2026[
    (predictions_2026["home_team"] == "Curaçao") |
    (predictions_2026["away_team"] == "Curaçao")
]

print(curacao_matches.to_string(index=False))

group   home_team away_team     M9_H     M9_D     M9_A    M11_H    M11_D    M11_A
    E     Germany   Curaçao 0.764516 0.155065 0.080419 0.756638 0.158991 0.084371
    E Ivory Coast   Curaçao 0.575181 0.236396 0.188422 0.569042 0.238043 0.192915
    E     Curaçao   Ecuador 0.077002 0.157877 0.765121 0.080894 0.162224 0.756882


In [131]:
# REEMPLAZAR CURAÇAO POR M3

for idx, row in fixtures_2026.iterrows():

    if (
        row["home_team"] == "Curaçao"
        or
        row["away_team"] == "Curaçao"
    ):

        p3 = predict_m3_symmetrized(
            model_M3_home,
            model_M3_away,
            row
        )

        predictions_2026.loc[idx, "M9_H"] = p3[0]
        predictions_2026.loc[idx, "M9_D"] = p3[1]
        predictions_2026.loc[idx, "M9_A"] = p3[2]

        predictions_2026.loc[idx, "M11_H"] = p3[0]
        predictions_2026.loc[idx, "M11_D"] = p3[1]
        predictions_2026.loc[idx, "M11_A"] = p3[2]

print("Fallback M3 aplicado a Curaçao.")

Fallback M3 aplicado a Curaçao.


### Simulación de variables

In [132]:
# MONTE CARLO COMPLETO — M9 Y M11

import numpy as np
import pandas as pd
from scipy.stats import poisson
from itertools import combinations


N_SIM = 5000
GMAX = 8







elo_month_lookup = (
    team_month_elo
    .set_index(["team", "month"])["elo_post"]
)

forecast_months = {
    0: pd.Period("2026-05", freq="M"),
    1: pd.Period("2026-04", freq="M"),
    2: pd.Period("2026-03", freq="M"),
    3: pd.Period("2026-02", freq="M"),
    4: pd.Period("2026-01", freq="M"),
    5: pd.Period("2025-12", freq="M"),
}


form_lookup = (
    form_2026
    .set_index("team")
)


def get_match_features(team1, team2):

    elo_diffs = []

    for lag in range(6):

        month = forecast_months[lag]

        e1 = elo_month_lookup.get(
            (team1, month),
            np.nan
        )

        e2 = elo_month_lookup.get(
            (team2, month),
            np.nan
        )

        elo_diffs.append(e1 - e2)

    return {
        "elo_diff_m0": elo_diffs[0],
        "elo_diff_m1": elo_diffs[1],
        "elo_diff_m2": elo_diffs[2],
        "elo_diff_m3": elo_diffs[3],
        "elo_diff_m4": elo_diffs[4],
        "elo_diff_m5": elo_diffs[5],

        "home_goals_for_6":
            form_lookup.loc[team1, "goals_for_6"],

        "home_goals_against_6":
            form_lookup.loc[team1, "goals_against_6"],

        "away_goals_for_6":
            form_lookup.loc[team2, "goals_for_6"],

        "away_goals_against_6":
            form_lookup.loc[team2, "goals_against_6"],

        "neutral": 1
    }






def get_sdr_scores_for_match(
    team1,
    team2,
    B,
    mean_2026,
    inv_sqrt_2026
):

    features = get_match_features(
        team1,
        team2
    )

    x = np.array([
        features[f"elo_diff_m{i}"]
        for i in range(6)
    ], dtype=float)

    x_white = transform_whitening(
        x.reshape(1, -1),
        mean_2026,
        inv_sqrt_2026
    )

    scores = x_white @ B

    return scores[0]






def poisson_matrix_from_models(
    team1,
    team2,
    model_home,
    model_away,
    score_cols,
    B,
    use_m3=False
):

    features = get_match_features(
        team1,
        team2
    )





    if use_m3:

        row = features

        p_original = get_score_matrix(
            model_home,
            model_away,
            pd.Series(row),
            ["elo_diff_m0"],
            gmax=GMAX
        )[2]

        reverse = features.copy()

        reverse["elo_diff_m0"] = (
            -features["elo_diff_m0"]
        )

        reverse["home_goals_for_6"] = (
            features["away_goals_for_6"]
        )

        reverse["home_goals_against_6"] = (
            features["away_goals_against_6"]
        )

        reverse["away_goals_for_6"] = (
            features["home_goals_for_6"]
        )

        reverse["away_goals_against_6"] = (
            features["home_goals_against_6"]
        )

        p_reverse = get_score_matrix(
            model_home,
            model_away,
            pd.Series(reverse),
            ["elo_diff_m0"],
            gmax=GMAX
        )[2]

    else:

        scores = get_sdr_scores_for_match(
            team1,
            team2,
            B,
            mean_2026,
            inv_sqrt_2026
        )

        row = features.copy()

        row[score_cols[0]] = scores[0]
        row[score_cols[1]] = scores[1]

        p_original = get_score_matrix(
            model_home,
            model_away,
            pd.Series(row),
            score_cols,
            gmax=GMAX
        )[2]


        reverse = features.copy()

        reverse_scores = -scores

        reverse[score_cols[0]] = reverse_scores[0]
        reverse[score_cols[1]] = reverse_scores[1]

        reverse["home_goals_for_6"] = (
            features["away_goals_for_6"]
        )

        reverse["home_goals_against_6"] = (
            features["away_goals_against_6"]
        )

        reverse["away_goals_for_6"] = (
            features["home_goals_for_6"]
        )

        reverse["away_goals_against_6"] = (
            features["home_goals_against_6"]
        )

        p_reverse = get_score_matrix(
            model_home,
            model_away,
            pd.Series(reverse),
            score_cols,
            gmax=GMAX
        )[2]





    matrix = 0.5 * (
        p_original +
        p_reverse.T
    )

    matrix /= matrix.sum()

    return matrix






all_teams = [
    team
    for group in groups_2026.values()
    for team in group
]

pair_keys = []

for i in range(len(all_teams)):
    for j in range(i + 1, len(all_teams)):
        pair_keys.append(
            (all_teams[i], all_teams[j])
        )


score_matrices_M9 = {}
score_matrices_M11 = {}


for team1, team2 in pair_keys:

    use_m3 = (
        team1 == "Curaçao"
        or
        team2 == "Curaçao"
    )

    if use_m3:

        matrix = poisson_matrix_from_models(
            team1,
            team2,
            model_M3_home,
            model_M3_away,
            ["elo_diff_m0"],
            None,
            use_m3=True
        )

        score_matrices_M9[
            (team1, team2)
        ] = matrix

        score_matrices_M11[
            (team1, team2)
        ] = matrix.copy()

    else:

        score_matrices_M9[
            (team1, team2)
        ] = poisson_matrix_from_models(
            team1,
            team2,
            model_M9_home,
            model_M9_away,
            ["sir_2_1", "sir_2_2"],
            B_sir_2026,
            use_m3=False
        )

        score_matrices_M11[
            (team1, team2)
        ] = poisson_matrix_from_models(
            team1,
            team2,
            model_M11_home,
            model_M11_away,
            ["save_2_1", "save_2_2"],
            B_save_2026,
            use_m3=False
        )


print(
    "Matrices M9:",
    len(score_matrices_M9)
)

print(
    "Matrices M11:",
    len(score_matrices_M11)
)

Matrices M9: 1128
Matrices M11: 1128


In [133]:
# Simulación de variables

def get_pair_matrix(
    team1,
    team2,
    matrices
):

    if (team1, team2) in matrices:

        return matrices[
            (team1, team2)
        ]


    return matrices[
        (team2, team1)
    ].T

In [134]:
# Simulación de variables

def sample_score(
    team1,
    team2,
    matrices,
    rng
):

    matrix = get_pair_matrix(
        team1,
        team2,
        matrices
    )

    flat = matrix.ravel()

    idx = rng.choice(
        len(flat),
        p=flat
    )

    g1, g2 = np.unravel_index(
        idx,
        matrix.shape
    )

    return int(g1), int(g2)

In [135]:
# Simulación de variables

def rank_group(
    group,
    teams,
    matches
):

    stats = {}

    for team in teams:

        stats[team] = {
            "points": 0,
            "gf": 0,
            "ga": 0,
            "gd": 0,
            "wins": 0,
            "draws": 0,
            "losses": 0
        }





    for m in matches:

        h = m["home_team"]
        a = m["away_team"]

        hg = m["home_goals"]
        ag = m["away_goals"]

        stats[h]["gf"] += hg
        stats[h]["ga"] += ag

        stats[a]["gf"] += ag
        stats[a]["ga"] += hg

        if hg > ag:

            stats[h]["points"] += 3
            stats[h]["wins"] += 1
            stats[a]["losses"] += 1

        elif hg < ag:

            stats[a]["points"] += 3
            stats[a]["wins"] += 1
            stats[h]["losses"] += 1

        else:

            stats[h]["points"] += 1
            stats[a]["points"] += 1

            stats[h]["draws"] += 1
            stats[a]["draws"] += 1

    for team in teams:

        stats[team]["gd"] = (
            stats[team]["gf"]
            -
            stats[team]["ga"]
        )





    def h2h_stats(tied):

        h2h = {
            t: {
                "points": 0,
                "gf": 0,
                "ga": 0
            }
            for t in tied
        }

        for m in matches:

            h = m["home_team"]
            a = m["away_team"]

            if h not in tied or a not in tied:
                continue

            hg = m["home_goals"]
            ag = m["away_goals"]

            h2h[h]["gf"] += hg
            h2h[h]["ga"] += ag

            h2h[a]["gf"] += ag
            h2h[a]["ga"] += hg

            if hg > ag:

                h2h[h]["points"] += 3

            elif hg < ag:

                h2h[a]["points"] += 3

            else:

                h2h[h]["points"] += 1
                h2h[a]["points"] += 1

        return h2h





    remaining = list(teams)
    ordered = []

    while remaining:

        max_points = max(
            stats[t]["points"]
            for t in remaining
        )

        tied = [
            t for t in remaining
            if stats[t]["points"] == max_points
        ]

        if len(tied) == 1:

            ordered.append(tied[0])
            remaining.remove(tied[0])
            continue

        h2h = h2h_stats(tied)


        max_h2h_points = max(
            h2h[t]["points"]
            for t in tied
        )

        tied2 = [
            t for t in tied
            if h2h[t]["points"] == max_h2h_points
        ]


        if len(tied2) > 1:

            tied2 = sorted(
                tied2,
                key=lambda t: (
                    h2h[t]["points"],
                    h2h[t]["gf"] - h2h[t]["ga"],
                    h2h[t]["gf"],
                    stats[t]["gd"],
                    stats[t]["gf"]
                ),
                reverse=True
            )

        else:

            tied2 = tied2



        tied2 = sorted(
            tied2,
            key=lambda t: (
                h2h[t]["points"],
                h2h[t]["gf"] - h2h[t]["ga"],
                h2h[t]["gf"],
                stats[t]["gd"],
                stats[t]["gf"]
            ),
            reverse=True
        )

        ordered.extend(tied2)

        for t in tied2:
            if t in remaining:
                remaining.remove(t)

    return ordered, stats

In [136]:
# Simulación de variables

def simulate_groups(
    matrices,
    rng
):

    all_group_matches = {}
    group_rankings = {}

    for group, teams in groups_2026.items():

        matches = []

        for i in range(4):

            for j in range(i + 1, 4):

                home = teams[i]
                away = teams[j]

                hg, ag = sample_score(
                    home,
                    away,
                    matrices,
                    rng
                )

                matches.append({
                    "group": group,
                    "home_team": home,
                    "away_team": away,
                    "home_goals": hg,
                    "away_goals": ag
                })

        ranking, stats = rank_group(
            group,
            teams,
            matches
        )

        all_group_matches[group] = matches
        group_rankings[group] = ranking

    return (
        all_group_matches,
        group_rankings
    )

In [137]:
# Simulación de variables

def get_qualifiers(
    group_rankings,
    all_group_matches
):

    first = {}
    second = {}
    thirds = []

    for group, ranking in group_rankings.items():

        first[group] = ranking[0]
        second[group] = ranking[1]


        _, stats = rank_group(
            group,
            groups_2026[group],
            all_group_matches[group]
        )

        thirds.append({
            "group": group,
            "team": ranking[2],
            "points": stats[ranking[2]]["points"],
            "gd": stats[ranking[2]]["gd"],
            "gf": stats[ranking[2]]["gf"]
        })

    thirds_df = pd.DataFrame(thirds)

    thirds_df = thirds_df.sort_values(
        ["points", "gd", "gf"],
        ascending=False
    ).reset_index(drop=True)

    best_thirds = thirds_df.head(8)

    return (
        first,
        second,
        best_thirds
    )

In [138]:
# Simulación de variables

third_slots = {
    73: set("ABCDF"),
    76: set("CDFGH"),
    78: set("CEFHI"),
    79: set("EHIJK"),
    80: set("BEFIJ"),
    81: set("AEHIJ"),
    84: set("EFGIJ"),
    86: set("DEIJL")
}

In [139]:
# Simulación de variables

def assign_thirds_to_slots(best_thirds):

    selected = list(
        best_thirds["group"]
    )

    slots = list(
        third_slots.keys()
    )

    solution = {}

    def backtrack(
        i,
        remaining,
        assignment
    ):

        if i == len(slots):
            return assignment.copy()

        slot = slots[i]

        possible = [
            g for g in remaining
            if g in third_slots[slot]
        ]



        possible = sorted(
            possible,
            key=lambda g: sum(
                g in third_slots[s]
                for s in slots[i:]
            )
        )

        for g in possible:

            assignment[slot] = g

            result = backtrack(
                i + 1,
                [
                    x for x in remaining
                    if x != g
                ],
                assignment
            )

            if result is not None:
                return result

            del assignment[slot]

        return None

    solution = backtrack(
        0,
        selected,
        {}
    )

    if solution is None:
        raise ValueError(
            "No se pudo asignar los terceros al R32."
        )

    return solution

In [140]:
# Simulación de variables

def build_r32(
    first,
    second,
    best_thirds
):

    third_assignment = assign_thirds_to_slots(
        best_thirds
    )

    third_team = {
        group: team
        for group, team in zip(
            best_thirds["group"],
            best_thirds["team"]
        )
    }

    r32 = [
        (72, second["A"], second["B"]),
        (73, first["E"], third_team[
            third_assignment[73]
        ]),
        (74, first["F"], second["C"]),
        (75, first["C"], second["F"]),
        (76, first["I"], third_team[
            third_assignment[76]
        ]),
        (77, second["E"], second["I"]),
        (78, first["A"], third_team[
            third_assignment[78]
        ]),
        (79, first["L"], third_team[
            third_assignment[79]
        ]),
        (80, first["D"], third_team[
            third_assignment[80]
        ]),
        (81, first["G"], third_team[
            third_assignment[81]
        ]),
        (82, second["K"], second["L"]),
        (83, first["H"], second["J"]),
        (84, first["B"], third_team[
            third_assignment[84]
        ]),
        (85, first["J"], second["H"]),
        (86, first["K"], third_team[
            third_assignment[86]
        ]),
        (87, second["D"], second["G"])
    ]

    return r32

In [141]:
# Simulación de variables

def simulate_knockout_match(
    team1,
    team2,
    matrices,
    rng
):

    g1, g2 = sample_score(
        team1,
        team2,
        matrices,
        rng
    )

    if g1 > g2:
        return team1

    if g2 > g1:
        return team2



    return rng.choice(
        [team1, team2]
    )

In [142]:
# R32

def simulate_knockout(
    r32,
    matrices,
    rng
):


    winners_r32 = []

    for match_id, team1, team2 in r32:

        winner = simulate_knockout_match(
            team1,
            team2,
            matrices,
            rng
        )

        winners_r32.append(winner)











    r16_pairs = [
        (winners_r32[2], winners_r32[5]),
        (winners_r32[1], winners_r32[3]),
        (winners_r32[4], winners_r32[6]),
        (winners_r32[7], winners_r32[8]),
        (winners_r32[11], winners_r32[12]),
        (winners_r32[9], winners_r32[10]),
        (winners_r32[14], winners_r32[0]),
        (winners_r32[13], winners_r32[15])
    ]



    r16_pairs[6] = (
        winners_r32[14],
        r16_pairs[0][0]
    )

    winners_r16 = []

    for team1, team2 in r16_pairs:

        winners_r16.append(
            simulate_knockout_match(
                team1,
                team2,
                matrices,
                rng
            )
        )







    qf_pairs = [
        (winners_r16[1], winners_r16[2]),
        (winners_r16[5], winners_r16[6]),
        (winners_r16[3], winners_r16[4]),
        (winners_r16[7], winners_r16[0])
    ]

    winners_qf = [
        simulate_knockout_match(
            a,
            b,
            matrices,
            rng
        )
        for a, b in qf_pairs
    ]


    sf_pairs = [
        (winners_qf[0], winners_qf[1]),
        (winners_qf[3], winners_qf[2])
    ]

    winners_sf = [
        simulate_knockout_match(
            a,
            b,
            matrices,
            rng
        )
        for a, b in sf_pairs
    ]


    champion = simulate_knockout_match(
        winners_sf[0],
        winners_sf[1],
        matrices,
        rng
    )

    return {
        "r32": winners_r32,
        "r16": winners_r16,
        "qf": winners_qf,
        "sf": winners_sf,
        "finalists": winners_sf,
        "champion": champion
    }

In [143]:
# PLANTILLA OFICIAL DEL ROUND OF 32

R32_TEMPLATE = {
    "M72": ("R_A", "R_B"),
    "M73": ("W_E", "3_A/B/C/D/F"),
    "M74": ("W_F", "R_C"),
    "M75": ("W_C", "R_F"),
    "M76": ("W_I", "3_C/D/F/G/H"),
    "M77": ("R_E", "R_I"),
    "M78": ("W_A", "3_C/E/F/H/I"),
    "M79": ("W_L", "3_E/H/I/J/K"),

    "M80": ("W_D", "3_B/E/F/I/J"),
    "M81": ("W_G", "3_A/E/H/I/J"),
    "M82": ("R_K", "R_L"),
    "M83": ("W_H", "R_J"),
    "M84": ("W_B", "3_E/F/G/I/J"),
    "M85": ("W_J", "R_H"),
    "M86": ("W_K", "3_D/E/I/J/L"),
    "M87": ("R_D", "R_G"),
}

R16_TEMPLATE = {
    "M88": ("M74", "M77"),
    "M89": ("M73", "M75"),
    "M90": ("M76", "M78"),
    "M91": ("M79", "M80"),
    "M92": ("M83", "M84"),
    "M93": ("M81", "M82"),
    "M94": ("M86", "M88"),
    "M95": ("M85", "M87"),
}

QF_TEMPLATE = {
    "M96": ("M89", "M90"),
    "M97": ("M93", "M94"),
    "M98": ("M91", "M92"),
    "M99": ("M95", "M96"),
}

SF_TEMPLATE = {
    "M100": ("M97", "M98"),
    "M101": ("M99", "M100"),
}

print("R32:", len(R32_TEMPLATE))
print("R16:", len(R16_TEMPLATE))
print("QF:", len(QF_TEMPLATE))
print("SF:", len(SF_TEMPLATE))

R32: 16
R16: 8
QF: 4
SF: 2


In [144]:
# Simulación de variables

print("fixtures_2026:", fixtures_2026.shape)
print("groups_2026:", len(groups_2026))

print("\nM9:")
print(model_M9_home)
print(model_M9_away)

print("\nM11:")
print(model_M11_home)
print(model_M11_away)

fixtures_2026: (72, 18)
groups_2026: 12

M9:

M11:


In [145]:
# PRUEBA: 100 TORNEOS COMPLETOS

N_TEST = 100

print("Iniciando prueba de simulación...")
print(f"Simulaciones: {N_TEST}")
print("Etapas: Grupos -> R32 -> R16 -> QF -> SF -> Final")

Iniciando prueba de simulación...
Simulaciones: 100
Etapas: Grupos -> R32 -> R16 -> QF -> SF -> Final


In [146]:
# Revisar objetos de probabilidades ya calculados

print("predictions_2026:", predictions_2026.shape)
print(predictions_2026.columns.tolist())

print("\nPrimeros partidos:")
display(predictions_2026.head())

predictions_2026: (72, 9)
['group', 'home_team', 'away_team', 'M9_H', 'M9_D', 'M9_A', 'M11_H', 'M11_D', 'M11_A']

Primeros partidos:


,group,home_team,away_team,M9_H,M9_D,M9_A,M11_H,M11_D,M11_A
0,A,Mexico,South Korea,0.395792,0.289419,0.314789,0.392911,0.289575,0.317514
1,A,Mexico,South Africa,0.595478,0.241161,0.163361,0.587685,0.244017,0.168298
2,A,Mexico,Czech Republic,0.466484,0.274618,0.258897,0.464862,0.274754,0.260384
3,A,South Korea,South Africa,0.532378,0.258011,0.209611,0.528023,0.259124,0.212854
4,A,South Korea,Czech Republic,0.401268,0.278372,0.320360,0.403164,0.277934,0.318902


In [147]:
# Simulación de variables

def symmetrize_probs(p_ab, p_ba):
    """
    p_ab:
        probabilidades cuando A aparece como local
        [P(H), P(D), P(A)]

    p_ba:
        probabilidades cuando B aparece como local
        [P(H), P(D), P(A)]

    Devuelve probabilidades desde la perspectiva
    del equipo A, con sede neutral.
    """

    p = np.array([
        0.5 * (p_ab[0] + p_ba[2]),
        0.5 * (p_ab[1] + p_ba[1]),
        0.5 * (p_ab[2] + p_ba[0])
    ])

    return p / p.sum()

In [148]:
# Simulación de variables

test = symmetrize_probs(
    np.array([0.60, 0.20, 0.20]),
    np.array([0.20, 0.20, 0.60])
)

print(test)
print("Suma:", test.sum())

[0.6 0.2 0.2]
Suma: 1.0


In [149]:
# Ver las variables relevantes que ya existen en memoria

vars_relevantes = [
    "model_M9_home",
    "model_M9_away",
    "model_M11_home",
    "model_M11_away",
    "model_M3_home",
    "model_M3_away",
    "team_month_elo",
    "form_2026",
    "B_sir_2026",
    "B_save_2026",
    "transform_whitening",
]

for v in vars_relevantes:
    print(f"{v}: {'✓ existe' if v in globals() else '✗ NO existe'}")

model_M9_home: ✓ existe
model_M9_away: ✓ existe
model_M11_home: ✓ existe
model_M11_away: ✓ existe
model_M3_home: ✓ existe
model_M3_away: ✓ existe
team_month_elo: ✓ existe
form_2026: ✓ existe
B_sir_2026: ✓ existe
B_save_2026: ✓ existe
transform_whitening: ✓ existe


In [150]:
# PREDICCIÓN DE UN PARTIDO NUEVO

def predict_pair(team_a, team_b, model="M9"):





    x = get_2026_elo_lag_vector(
        team_a,
        team_b
    )

    if np.isnan(x).any():
        raise ValueError(
            f"Elo incompleto para {team_a} vs {team_b}: {x}"
        )





    x_centered = x - X_mean

    eigenvalues, eigenvectors = np.linalg.eigh(X_cov)

    eigenvalues = np.maximum(
        eigenvalues,
        1e-10
    )

    inv_sqrt = (
        eigenvectors
        @ np.diag(1 / np.sqrt(eigenvalues))
        @ eigenvectors.T
    )

    x_white = x_centered @ inv_sqrt





    if model == "M9":

        z = x_white @ B_sir_2

        score_cols = [
            "sir_2_1",
            "sir_2_2"
        ]

        model_home = model_M9_home
        model_away = model_M9_away

    elif model == "M11":

        z = x_white @ B_save_2

        score_cols = [
            "save_2_1",
            "save_2_2"
        ]

        model_home = model_M11_home
        model_away = model_M11_away

    else:

        raise ValueError(
            f"Modelo no reconocido: {model}"
        )

    z1 = float(z[0])
    z2 = float(z[1])





    form_h = form_2026.loc[
    form_2026["team"] == team_a
           ].iloc[0]

    form_a = form_2026.loc[
    form_2026["team"] == team_b
      ].iloc[0]





    neutral = 1

    home_goals_for = form_a["goals_for_6"]
    away_goals_against = form_h["goals_against_6"]

    away_goals_for = form_h["goals_for_6"]
    home_goals_against = form_a["goals_against_6"]

    X_home = pd.DataFrame({
        score_cols[0]: [z1],
        score_cols[1]: [z2],
        "neutral": [neutral],
        "home_goals_for_6": [home_goals_for],
        "away_goals_against_6": [away_goals_against]
    })

    X_home = sm.add_constant(
        X_home,
        has_constant="add"
    )

    X_away = pd.DataFrame({
        score_cols[0]: [-z1],
        score_cols[1]: [-z2],
        "neutral": [neutral],
        "away_goals_for_6": [away_goals_for],
        "home_goals_against_6": [home_goals_against]
    })

    X_away = sm.add_constant(
        X_away,
        has_constant="add"
    )





    lambda_home = float(
        model_home.predict(X_home)[0]
    )

    lambda_away = float(
        model_away.predict(X_away)[0]
    )





    matrix = np.zeros((9, 9))

    for gh in range(9):

        for ga in range(9):

            matrix[gh, ga] = (
                poisson.pmf(
                    gh,
                    lambda_home
                )
                *
                poisson.pmf(
                    ga,
                    lambda_away
                )
            )


    matrix /= matrix.sum()





    p_home = np.tril(
        matrix,
        k=-1
    ).sum()

    p_draw = np.trace(matrix)

    p_away = np.triu(
        matrix,
        k=1
    ).sum()

    probs = np.array([
        p_home,
        p_draw,
        p_away
    ])

    probs /= probs.sum()

    return {
        "lambda_home": lambda_home,
        "lambda_away": lambda_away,
        "matrix": matrix,
        "probs": probs,
        "z": np.array([
            z1,
            z2
        ])
    }

In [151]:
# Simulación de variables

def get_2026_elo_lag_vector(team1, team2):
    elo_diffs = []
    for lag in range(6):
        month = forecast_months[lag]
        e1 = elo_month_lookup.get((team1, month), np.nan)
        e2 = elo_month_lookup.get((team2, month), np.nan)
        elo_diffs.append(e1 - e2)
    return np.array(elo_diffs, dtype=float)

In [152]:
# PROBABILIDAD NEUTRAL PARA EL MOTOR DEL TORNEO

def neutral_probability(team_a, team_b, model="M9"):

    result = predict_pair(
        team_a,
        team_b,
        model
    )

    return np.asarray(
        result["probs"],
        dtype=float
    )

In [153]:
# Simulación de variables

form_2026["team"].tolist()

['Mexico',
 'South Korea',
 'South Africa',
 'Czech Republic',
 'Canada',
 'Bosnia and Herzegovina',
 'Qatar',
 'Switzerland',
 'Brazil',
 'Morocco',
 'Haiti',
 'Scotland',
 'United States',
 'Paraguay',
 'Australia',
 'Turkey',
 'Germany',
 'Ivory Coast',
 'Curaçao',
 'Ecuador',
 'Netherlands',
 'Japan',
 'Sweden',
 'Tunisia',
 'Belgium',
 'Egypt',
 'Iran',
 'New Zealand',
 'Spain',
 'Cape Verde',
 'Saudi Arabia',
 'Uruguay',
 'France',
 'Senegal',
 'Iraq',
 'Norway',
 'Argentina',
 'Algeria',
 'Austria',
 'Jordan',
 'Portugal',
 'DR Congo',
 'Uzbekistan',
 'Colombia',
 'England',
 'Croatia',
 'Ghana',
 'Panama']

In [154]:
# Simulación de variables

form_2026[form_2026["team"].astype(str).str.contains("Mexico", case=False, na=False)]

,team,goals_for_6,goals_against_6
0,Mexico,1.5,0.166667


In [155]:
# TORNEO COMPLETO: GRUPOS -> R32 -> R16 -> QF -> SF -> FINAL

RNG = np.random.default_rng(42)

from collections import defaultdict











groups = groups_2026.copy()


groups = {
    str(g): list(teams)
    for g, teams in groups.items()
}






def play_group_match(team_a, team_b, model):

    p = neutral_probability(team_a, team_b, model)

    outcome = RNG.choice(
        [0, 1, 2],
        p=p
    )





    if outcome == 0:
        return team_a, team_b, 3, 0, 1, 0

    elif outcome == 1:
        return team_a, team_b, 1, 1, 0, 0

    else:
        return team_a, team_b, 0, 3, 0, 1






def simulate_groups(model):

    standings = {}
    group_matches = {}

    for group, teams in groups.items():

        stats = {}

        for team in teams:

            stats[team] = {
                "team": team,
                "pts": 0,
                "gf": 0,
                "ga": 0,
                "gd": 0,
                "wins": 0,
                "draws": 0,
                "losses": 0
            }

        matches = []


        for i in range(len(teams)):

            for j in range(i + 1, len(teams)):

                a = teams[i]
                b = teams[j]

                p = neutral_probability(a, b, model)

                outcome = RNG.choice(
                    ["H", "D", "A"],
                    p=p
                )





                if outcome == "H":


                    while True:

                        gh = RNG.poisson(1.5)
                        ga = RNG.poisson(0.8)

                        if gh > ga:
                            break

                    stats[a]["pts"] += 3
                    stats[a]["wins"] += 1
                    stats[b]["losses"] += 1

                elif outcome == "D":


                    g = RNG.poisson(1.0)

                    gh = g
                    ga = g

                    stats[a]["pts"] += 1
                    stats[b]["pts"] += 1

                    stats[a]["draws"] += 1
                    stats[b]["draws"] += 1

                else:

                    while True:

                        gh = RNG.poisson(0.8)
                        ga = RNG.poisson(1.5)

                        if ga > gh:
                            break

                    stats[b]["pts"] += 3
                    stats[b]["wins"] += 1
                    stats[a]["losses"] += 1

                stats[a]["gf"] += gh
                stats[a]["ga"] += ga

                stats[b]["gf"] += ga
                stats[b]["ga"] += gh

                matches.append(
                    (a, b, gh, ga)
                )





        for team in teams:

            stats[team]["gd"] = (
                stats[team]["gf"]
                -
                stats[team]["ga"]
            )





        ranked = sorted(
            stats.values(),
            key=lambda x: (
                x["pts"],
                x["gd"],
                x["gf"]
            ),
            reverse=True
        )

        standings[group] = ranked
        group_matches[group] = matches

    return standings, group_matches






def get_qualifiers(standings):

    qualified = {}

    third_place = []

    for group, table in standings.items():

        qualified[group] = {
            "first": table[0]["team"],
            "second": table[1]["team"],
            "third": table[2]["team"]
        }

        third_place.append(
            (
                group,
                table[2]["team"],
                table[2]["pts"],
                table[2]["gd"],
                table[2]["gf"]
            )
        )


    third_place = sorted(
        third_place,
        key=lambda x: (
            x[2],
            x[3],
            x[4]
        ),
        reverse=True
    )

    best_thirds = third_place[:8]

    return qualified, best_thirds






R32_ALLOWED = {

    "M73": ["A", "B", "C", "D", "F"],

    "M76": ["C", "D", "F", "G", "H"],

    "M78": ["C", "E", "F", "H", "I"],

    "M79": ["E", "H", "I", "J", "K"],

    "M80": ["B", "E", "F", "I", "J"],

    "M81": ["A", "E", "H", "I", "J"],

    "M84": ["E", "F", "G", "I", "J"],

    "M86": ["D", "E", "I", "J", "L"]
}


def assign_thirds(best_thirds):

    groups_third = {
        x[0]: x[1]
        for x in best_thirds
    }

    slots = list(R32_ALLOWED.keys())


    def backtrack(i, used, assignment):

        if i == len(slots):
            return assignment.copy()

        slot = slots[i]

        for g in R32_ALLOWED[slot]:

            if g in groups_third and g not in used:

                assignment[slot] = groups_third[g]

                result = backtrack(
                    i + 1,
                    used | {g},
                    assignment
                )

                if result is not None:
                    return result

                del assignment[slot]

        return None

    result = backtrack(
        0,
        set(),
        {}
    )

    if result is None:

        raise ValueError(
            "No se encontró asignación válida de terceros."
        )

    return result






def knockout_match(team_a, team_b, model):

    p = neutral_probability(
        team_a,
        team_b,
        model
    )

    outcome = RNG.choice(
        ["A", "D", "B"],
        p=p
    )

    if outcome == "A":
        return team_a

    if outcome == "B":
        return team_b


    return RNG.choice(
        [team_a, team_b]
    )






def simulate_tournament(model="M9"):





    standings, matches = simulate_groups(model)

    qualified, best_thirds = get_qualifiers(
        standings
    )

    third_assignment = assign_thirds(
        best_thirds
    )






    r32 = {}

    r32["M72"] = knockout_match(
        qualified["A"]["second"],
        qualified["B"]["second"],
        model
    )

    r32["M73"] = knockout_match(
        qualified["E"]["first"],
        third_assignment["M73"],
        model
    )

    r32["M74"] = knockout_match(
        qualified["F"]["first"],
        qualified["C"]["second"],
        model
    )

    r32["M75"] = knockout_match(
        qualified["C"]["first"],
        qualified["F"]["second"],
        model
    )

    r32["M76"] = knockout_match(
        qualified["I"]["first"],
        third_assignment["M76"],
        model
    )

    r32["M77"] = knockout_match(
        qualified["E"]["second"],
        qualified["I"]["second"],
        model
    )

    r32["M78"] = knockout_match(
        qualified["A"]["first"],
        third_assignment["M78"],
        model
    )

    r32["M79"] = knockout_match(
        qualified["L"]["first"],
        third_assignment["M79"],
        model
    )

    r32["M80"] = knockout_match(
        qualified["D"]["first"],
        third_assignment["M80"],
        model
    )

    r32["M81"] = knockout_match(
        qualified["G"]["first"],
        third_assignment["M81"],
        model
    )

    r32["M82"] = knockout_match(
        qualified["K"]["second"],
        qualified["L"]["second"],
        model
    )

    r32["M83"] = knockout_match(
        qualified["H"]["first"],
        qualified["J"]["second"],
        model
    )

    r32["M84"] = knockout_match(
        qualified["B"]["first"],
        third_assignment["M84"],
        model
    )

    r32["M85"] = knockout_match(
        qualified["J"]["first"],
        qualified["H"]["second"],
        model
    )

    r32["M86"] = knockout_match(
        qualified["K"]["first"],
        third_assignment["M86"],
        model
    )

    r32["M87"] = knockout_match(
        qualified["D"]["second"],
        qualified["G"]["second"],
        model
    )






    r16 = {}


    r16["M88"] = knockout_match(
        r32["M73"],
        r32["M76"],
        model
    )


    r16["M89"] = knockout_match(
        r32["M72"],
        r32["M74"],
        model
    )


    r16["M90"] = knockout_match(
        r32["M75"],
        r32["M77"],
        model
    )


    r16["M91"] = knockout_match(
        r32["M78"],
        r32["M79"],
        model
    )


    r16["M92"] = knockout_match(
        r32["M82"],
        r32["M83"],
        model
    )


    r16["M93"] = knockout_match(
        r32["M80"],
        r32["M81"],
        model
    )


    r16["M94"] = knockout_match(
        r32["M85"],
        r32["M87"],
        model
    )


    r16["M95"] = knockout_match(
        r32["M84"],
        r32["M86"],
        model
    )






    qf = {}


    qf["M96"] = knockout_match(
        r16["M88"],
        r16["M89"],
        model
    )


    qf["M97"] = knockout_match(
        r16["M92"],
        r16["M93"],
        model
    )


    qf["M98"] = knockout_match(
        r16["M90"],
        r16["M91"],
        model
    )


    qf["M99"] = knockout_match(
        r16["M94"],
        r16["M95"],
        model
    )






    sf = {}


    sf["SF1"] = knockout_match(
        qf["M96"],
        qf["M97"],
        model
    )


    sf["SF2"] = knockout_match(
        qf["M98"],
        qf["M99"],
        model
    )






    champion = knockout_match(
        sf["SF1"],
        sf["SF2"],
        model
    )






    return {
        "standings": standings,
        "best_thirds": best_thirds,
        "r32": r32,
        "r16": r16,
        "qf": qf,
        "sf": sf,
        "champion": champion
    }






print("Ejecutando 1 torneo de prueba...")

test_tournament = simulate_tournament("M9")

print(
    "Campeón de prueba:",
    test_tournament["champion"]
)

print("OK")

Ejecutando 1 torneo de prueba...
Campeón de prueba: Colombia
OK


In [156]:
# MODELO SIR

N_SIMULATIONS = 100

results_M9 = []

print(f"Ejecutando {N_SIMULATIONS} simulaciones M9...")

for i in range(N_SIMULATIONS):
    results_M9.append(simulate_tournament("M9"))

print("M9 listo.")



results_M11 = []

print(f"Ejecutando {N_SIMULATIONS} simulaciones M11...")

for i in range(N_SIMULATIONS):
    results_M11.append(simulate_tournament("M11"))

print("M11 listo.")

Ejecutando 100 simulaciones M9...
M9 listo.
Ejecutando 100 simulaciones M11...
M11 listo.


In [157]:
# CAMPEONES

champions_M9 = Counter(r["champion"] for r in results_M9)
champions_M11 = Counter(r["champion"] for r in results_M11)

print("M9:")
print(champions_M9.most_common())

print("\nM11:")
print(champions_M11.most_common())


def championship_percentages(results):
    counts = Counter(r["champion"] for r in results)

    return {
        team: round(100 * n / len(results), 2)
        for team, n in counts.most_common()
    }

print("M9")
print(championship_percentages(results_M9))

print("\nM11")
print(championship_percentages(results_M11))

M9:
[(np.str_('Argentina'), 14), (np.str_('Brazil'), 14), ('Spain', 12), (np.str_('France'), 11), (np.str_('Netherlands'), 6), (np.str_('Uruguay'), 5), (np.str_('Colombia'), 4), ('England', 4), (np.str_('Switzerland'), 4), ('Portugal', 3), (np.str_('Ecuador'), 3), ('Mexico', 3), (np.str_('Morocco'), 2), (np.str_('Croatia'), 2), (np.str_('Japan'), 2), (np.str_('Belgium'), 2), ('Panama', 1), (np.str_('Paraguay'), 1), (np.str_('Sweden'), 1), (np.str_('Ivory Coast'), 1), (np.str_('Egypt'), 1), (np.str_('Australia'), 1), (np.str_('South Korea'), 1), (np.str_('Turkey'), 1), (np.str_('Canada'), 1)]

M11:
[(np.str_('Argentina'), 14), (np.str_('Spain'), 14), (np.str_('France'), 13), (np.str_('Ecuador'), 5), (np.str_('England'), 5), (np.str_('Portugal'), 4), (np.str_('Uruguay'), 4), (np.str_('Japan'), 4), (np.str_('Colombia'), 4), (np.str_('Brazil'), 4), (np.str_('Switzerland'), 3), (np.str_('Mexico'), 3), (np.str_('Croatia'), 3), (np.str_('Netherlands'), 3), (np.str_('Morocco'), 2), ('South Kor

In [158]:
# BRACKETS

test = results_M9[0]

print(test.keys())
print(test)

dict_keys(['standings', 'best_thirds', 'r32', 'r16', 'qf', 'sf', 'champion'])
{'standings': {'A': [{'team': 'South Korea', 'pts': 7, 'gf': 8, 'ga': 4, 'gd': 4, 'wins': 2, 'draws': 1, 'losses': 0}, {'team': 'Czech Republic', 'pts': 3, 'gf': 4, 'ga': 4, 'gd': 0, 'wins': 0, 'draws': 3, 'losses': 0}, {'team': 'Mexico', 'pts': 2, 'gf': 3, 'ga': 5, 'gd': -2, 'wins': 0, 'draws': 2, 'losses': 1}, {'team': 'South Africa', 'pts': 2, 'gf': 3, 'ga': 5, 'gd': -2, 'wins': 0, 'draws': 2, 'losses': 1}], 'B': [{'team': 'Canada', 'pts': 9, 'gf': 10, 'ga': 2, 'gd': 8, 'wins': 3, 'draws': 0, 'losses': 0}, {'team': 'Switzerland', 'pts': 6, 'gf': 8, 'ga': 3, 'gd': 5, 'wins': 2, 'draws': 0, 'losses': 1}, {'team': 'Qatar', 'pts': 3, 'gf': 3, 'ga': 6, 'gd': -3, 'wins': 1, 'draws': 0, 'losses': 2}, {'team': 'Bosnia and Herzegovina', 'pts': 0, 'gf': 0, 'ga': 10, 'gd': -10, 'wins': 0, 'draws': 0, 'losses': 3}], 'C': [{'team': 'Morocco', 'pts': 9, 'gf': 5, 'ga': 0, 'gd': 5, 'wins': 3, 'draws': 0, 'losses': 0}, {'t

In [159]:
# RESUMEN DE CAMPEONES

def champion_summary(results, model):
    counts = Counter(str(r["champion"]) for r in results)
    n = len(results)

    rows = [
        {
            "Modelo": model,
            "Equipo": team,
            "Campeonatos": count,
            "Porcentaje": round(100 * count / n, 2)
        }
        for team, count in counts.most_common()
    ]

    return pd.DataFrame(rows)


df_m9 = champion_summary(results_M9, "M9")
df_m11 = champion_summary(results_M11, "M11")





def extract_bracket(result, model):

    rows = []

    for stage in ["r32", "r16", "qf"]:

        for match, winner in result.get(stage, {}).items():

            rows.append({
                "Modelo": model,
                "Fase": stage.upper(),
                "Partido": match,
                "Ganador": str(winner)
            })

    for i, winner in enumerate(result.get("sf", []), 1):

        rows.append({
            "Modelo": model,
            "Fase": "SEMIFINAL",
            "Partido": f"SF{i}",
            "Ganador": str(winner)
        })

    rows.append({
        "Modelo": model,
        "Fase": "FINAL",
        "Partido": "FINAL",
        "Ganador": str(result["champion"])
    })

    return pd.DataFrame(rows)


bracket_m9 = extract_bracket(results_M9[0], "M9")
bracket_m11 = extract_bracket(results_M11[0], "M11")





archivo = "resultados_mundial_simulaciones.xlsx"

with pd.ExcelWriter(archivo, engine="openpyxl") as writer:


    df_m9.to_excel(
        writer,
        sheet_name="Campeones_M9",
        index=False
    )


    df_m11.to_excel(
        writer,
        sheet_name="Campeones_M11",
        index=False
    )


    bracket_m9.to_excel(
        writer,
        sheet_name="Bracket_M9",
        index=False
    )


    bracket_m11.to_excel(
        writer,
        sheet_name="Bracket_M11",
        index=False
    )

print(f"Excel creado: {archivo}")

Excel creado: resultados_mundial_simulaciones.xlsx
